# Category Matrix – attribute assignment (pilot: **Akkuleuchten**)

Repeatable workflow: read source files → normalise → select products → join attributes and images → assign every attribute from the allowed list → validate → export.

**How the attributes are assigned (Akkuleuchten)**

| Attribute | Method |
|---|---|
| Design style | Taken from the specialist file `DesignStyle_Akkuleuchte.csv` for labelled products. All other products get the label of the **most similar labelled product** (text + structured data + image similarity). |
| Colour Group, Material & look, Light control, Environment | Rules with a **fixed evidence priority**: (1) product attributes (`ItemAttribute.csv`, structured product fields) → (2) product text (name, variant) → (3) product photo (image model / most similar photo) → (4) optional default. A lower level is only used when the higher levels give nothing. |

Only the cells in **Section 0** normally need to be edited. Output is kept short; details go to the log file in `RES`.

In [ ]:
# =====================================================================
# 0. CONFIGURATION  (edit here – the main logic below does not need changes)
# =====================================================================
from pathlib import Path

# ---- 0.1 Folders --------------------------------------------------------
PROJECT_DIR = Path.cwd()            # the folder "Category Matrix" (where this notebook lives)
SRC_DIR     = PROJECT_DIR / "SRC"   # source files
RES_DIR     = PROJECT_DIR / "RES"   # result files
CACHE_DIR   = RES_DIR / "cache_images"   # downloaded + validated images
INTERIM_DIR = RES_DIR / "intermediate"   # audit files (neighbours, image log, ...)

# ---- 0.2 Source files: found by name pattern (case, spaces, symbols are ignored) ----
FILE_PATTERNS = {
    "products":   "dwh_present_select",
    "media":      "ItemMedia",
    "attributes": "ItemAttribute",
    "sales":      "Orders sales",                       # "1. Group_ Orders & sales.csv"
    "allowed":    "category_attribute_values_vertical",
}

# ---- 0.3 Categories to process ----------------------------------------------
# dwh_subcategory : value(s) in the product file column "ShopifySubCategory"
# allowed_category: name in category_attribute_values_vertical.csv (partial match is accepted)
# label_file      : name pattern of the specialist label file (None = no labels yet)
CATEGORIES = {
    "Akkuleuchten": {"dwh_subcategory": ["Akkuleuchten"], "allowed_category": "Akkuleuchten",
                     "label_file": "DesignStyle_Akkuleuchte"},
    # --- switch on later by removing the '#' (add a label file per category when available) ---
    # "Vase":         {"dwh_subcategory": ["Vasen"],        "allowed_category": "Vase",         "label_file": None},
    # "Esstisch":     {"dwh_subcategory": ["Esstische"],    "allowed_category": "Esstisch",     "label_file": None},
    # "Loungesessel": {"dwh_subcategory": ["Loungesessel"], "allowed_category": "Loungesessel", "label_file": None},
    # "Regalsysteme": {"dwh_subcategory": ["Regalsysteme"], "allowed_category": "Regalsysteme", "label_file": None},
    # "Tassen":       {"dwh_subcategory": ["Tassen"],       "allowed_category": "Tassen",       "label_file": None},
}
# Labelled products that belong to another sub-category in the product file (e.g. a "Stehleuchte")
# are still used as similarity references (never exported as target products).
REFERENCE_FROM_ANY_SUBCATEGORY = True

# ---- 0.4 Column mappings (first matching name wins; matching ignores case/spaces/symbols) ----
COLUMN_MAP = {
    "products": {"item_id": ["ItemId"], "name": ["Name"], "subcategory": ["ShopifySubCategory"],
                 "main_category": ["ShopifyMainCategory"], "top_category": ["ShopifyTopCategory"],
                 "manufacturer_id": ["ManufacturerId"], "length": ["Length"], "depth": ["Depth"],
                 "height": ["Height"], "weight": ["Weight"], "variant": ["Variant"],
                 "colour": ["ColorName", "ColourName"], "material": ["Material"]},
    "media":      {"item_id": ["ItemId"], "media_id": ["MediaId"], "order": ["Order"], "url": ["PartialPath", "Url"]},
    "attributes": {"item_id": ["ItemId"], "set": ["AttributeSetName"], "name": ["AttributeName"], "value": ["AttributeValue"]},
    "allowed":    {"category": ["Category"], "attribute": ["Attribute"], "value": ["Value"]},
    "labels":     {"item_id": ["ItemId"], "attribute": ["Attribute"], "value": ["AttributeValue", "Value"]},
    # sales: 'date' and 'order_id' are optional – they are used automatically if such a column appears later
    "sales":      {"item_id": ["Item (group)", "ItemId"], "item_name": ["Unnamed: 1", "Item name"],
                   "date": ["Date", "Order date", "Posting date", "Datum", "Month"],
                   "order_id": ["Order id", "Order number", "Order no", "Orderno"]},
}
# ---- 0.5 Sales ---------------------------------------------------------------
SALES_PERIOD = ("2025-11-01", "2026-10-31")      # inclusive; only applied if a date column exists
SALES_CURRENCY = "not stated in source file (assumed EUR)"
# output name: (candidate column names, aggregation "sum" or "first")
SALES_MEASURES = {
    "net_order_intake_I":  (["Net order intake I (group)"], "sum"),
    "order_item_qty":      (["Order item qty (group)"], "sum"),
    "net_order_intake_II": (["Net order intake II (group)"], "sum"),
    "noi_II_item_qty":     (["NOI II item qty (group)"], "sum"),
    "stock_qty_available": (["Stock qty - available"], "first"),   # stock is a snapshot, not summed
}
SALES_PRIMARY_REVENUE = "net_order_intake_II"    # used for ranking / top sellers
SALES_PRIMARY_UNITS   = "noi_II_item_qty"

# ---- 0.6 Images --------------------------------------------------------------
IMAGE_SETTINGS = {
    "download": True,             # False = use only already cached images
    "max_images_per_product": 1,  # first image by 'Order' (packshot); raise to try fallbacks
    "skip_order_values": [999999],# 'Order' values that are not product photos
    "base_url": "",               # prefix if PartialPath contains relative paths
    "connect_timeout": 5, "read_timeout": 15, "retries": 2, "workers": 8,
    "max_side": 512, "min_side": 64,
    "abort_after_failures": 25,   # stop early if the first N downloads all fail (e.g. no internet/proxy)
    "user_agent": "CategoryMatrixNotebook/1.0",
}
IMAGE_FEATURES = {"bg_tolerance": 28}   # colour distance to the border colour treated as background

# ---- 0.7 Optional model (image embeddings + zero-shot material) – needs torch + transformers ----
USE_CLIP = True        # needs: torch + transformers (downloads the model once). If missing, the notebook continues without it.
CLIP_MODEL = "openai/clip-vit-base-patch32"

# ---- 0.7b Photo fallback: used when an attribute has neither attribute nor text evidence ----
IMAGE_FALLBACK = {
    "enabled": True,
    "min_image_similarity": 0.80,     # 'most similar photo' must reach this value (0-1) to be used at all
    "medium_image_similarity": 0.92,  # at/above: Medium confidence, below: Low
    "clip_min_prob": 0.40,            # zero-shot image model: minimum probability of the top value
    "clip_medium_prob": 0.70,         # at/above: Medium confidence, below: Low
    "require_beat_baseline": True,    # a photo signal is only used for an attribute if, on products that already have a
                                      # value, it reproduces that value more often than 'always the most common value'
}

# ---- 0.8 Similarity for label transfer ---------------------------------------
SIMILARITY = {
    "weights": {"text": 0.45, "structured": 0.20, "image": 0.35},  # components missing for a pair are ignored; tune with the check table
    "k": 1,                       # 1 = take the label of the single most similar labelled product
    "high": 0.80, "medium": 0.65, # similarity thresholds for High / Medium confidence
    "min_margin_for_high": 0.03,  # best similarity must beat the best other-label similarity by this much
}
NUMERIC_FEATURES     = ["length", "depth", "height", "weight"]       # structured similarity
CATEGORICAL_FEATURES = ["manufacturer_id", "product_type", "series"]

# ---- 0.9 Add NEW information later (tier, price, ...) -------------------------------------
# role: "numeric" / "categorical" / "text" -> used in similarity;  "info" -> only carried to the outputs
EXTRA_PRODUCT_FILES = [
    # {"file_pattern": "tier_price", "key": ["ItemId"], "columns": {"Tier": "categorical", "Price": "numeric"}},
]
EXTRA_ATTRIBUTE_FEATURES = {   # take a value from ItemAttribute.csv as an extra product column
    # "battery_mah": {"attribute": "Akku-Kapazität", "role": "numeric"},
}

# ---- 0.10 Output / behaviour -----------------------------------------------------
NOT_DETERMINABLE = "Not determinable"   # placeholder when no valid value can be supported (see README)
STRICT_VALIDATION = False               # True = raise an error when a validation check fails
MAX_NOT_DETERMINABLE_SHARE = 0.30       # warn if an attribute has more than this share of placeholders
CSV_SEP, CSV_ENCODING = ";", "utf-8-sig"   # ';' + BOM opens correctly in German Excel
OUTPUT_FILES = {
    "detail":   "assignments_detailed.csv",
    "wide":     "products_wide.csv",
    "sales":    "sales_summary_by_product.csv",
    "wide_sales": "products_wide_with_sales.csv",
    "validation": "validation_report.csv",
    "log":      "processing_log.txt",
}

In [ ]:
# ---- 0.11 Assignment rules (editable). Keywords are matched case/accent-insensitively. ----
# Prefix '=' means "whole word only" (e.g. '=app' does not match 'apple').

# Colour words -> colour group (must match the allowed values of the category)
COLOUR_KEYWORDS = {
    "Light Neutrals": ["weiss", "white", "creme", "cream", "elfenbein", "ivory", "beige", "sand", "nude",
                       "champagner", "putty", "greige", "perlmutt", "vanille", "alabaster", "offwhite", "pearl", "perle", "milk",
                       "milch", "cloud", "wheat", "latte", "puder", "leinen", "linen", "snow", "schnee", "chalk", "kreide", "mandel", "almond", "cashmere", "blanc"],
    "Warm Earth & Natural Woods": ["braun", "brown", "terracotta", "terrakotta", "ziegelrot", "rost", "rust",
                       "messing", "brass", "gold", "kupfer", "copper", "bronze", "holz", "wood", "eiche", "oak",
                       "walnuss", "walnut", "nussbaum", "natur", "natural", "camel", "cognac", "taupe", "ocker",
                       "caramel", "karamell", "chocolate", "schokolade", "bernstein", "amber", "lebkuchen", "honig", "honey",
                       "espresso", "mocha", "lehm", "clay", "maple", "sienna", "ginger", "kelp", "sahara", "dune", "olive", "oliv", "khaki", "cappuccino", "olivgrun", "olivgruen"],
    "Cool Grays & Minerals": ["grau", "grey", "gray", "silber", "silver", "chrom", "chrome", "edelstahl", "stahl",
                       "steel", "aluminium", "aluminum", "alu", "stein", "stone", "beton", "concrete", "nickel",
                       "titan", "zinn", "blaugrau", "graublau", "hellgrau", "platin", "mineral", "gris", "mist", "slate", "schiefer",
                       "basalt", "asche", "ash", "granite", "granit", "rauch", "smoke", "gun"],
    "Deep Shadows & Graphites": ["schwarz", "black", "anthrazit", "anthracite", "graphit", "graphite", "charcoal",
                       "dunkelgrau", "onyx", "nachtschwarz", "lakritze", "noir", "midnight", "nacht"],
    "Chromatic Accents & Jewel Tones": ["rot", "red", "blau", "blue", "gruen", "grun", "green", "gelb", "yellow",
                       "orange", "pink", "rosa", "rose", "lila", "violett", "purple", "violet", "tuerkis", "turkis",
                       "turquoise", "burgundy", "bordeaux", "petrol", "coral", "koralle", "mint", "ruby", "beetroot",
                       "magenta", "lavendel", "lavender", "aubergine", "smaragd", "emerald", "saphir", "sapphire",
                       "navy", "marine", "cyan", "lime", "apricot", "peach", "salbei", "sage", "senf", "mustard", "pflaume", "plum",
                       "zitrone", "lemon", "maroon", "grape", "kirsch", "cherry", "tomato", "himbeere", "raspberry",
                       "ocean", "sky", "denim", "indigo", "kobalt", "cobalt", "jade", "moss", "moos", "teal", "glyzinie",
                       "wisteria", "fuchsia", "flieder", "lilac", "mandarine", "tangerine", "pistazie", "pistachio", "limoncello",
                       "cayenne", "kaktus", "fluoro", "sunflower", "leaf", "minze", "gletscherminze", "schwarzkirsche"],
}
COLOUR_MIX_VALUE = "Mix"                       # used when different colour groups are named together
COLOUR_COMPOUND_BASES = ["weiss", "schwarz", "grau", "blau", "gruen", "grun", "rot", "gelb", "braun", "rosa",
                         "orange", "lila", "beige", "silber", "gold"]     # e.g. 'cremeweiss', 'hellblau'

# =====================================================================================
# EVIDENCE PRIORITY (fixed in the code, see README):
#   1 product attributes  ->  2 product text  ->  3 photo  ->  4 default
# A lower level is only used when the levels above it deliver nothing for this attribute.
# =====================================================================================

# ---------- Material & look ----------
# (a) Attribute families in ItemAttribute.csv: "Material (Metall)", "Material 2 (Kunststoff)", ...
#     family word (in brackets) -> allowed value.  None = decide by the value text (MATERIAL_KEYWORDS)
MATERIAL_ATTRIBUTE_FAMILIES = {"metall": "Metal", "kunststoff": "Plastic/silicone", "holz": "Other",
                               "textil": "Fabric shade", "mineralien": None, "verschiedene": None}
MATERIAL_FAMILY_TEXT_FALLBACK = {"mineralien": "Other"}   # used when the value text is not in MATERIAL_KEYWORDS
# (b) Yes/no flags (value True): "Metall", "Kunststoff", "Glas", ...
MATERIAL_FLAG_ATTRIBUTES = {"Metall": "Metal", "Kunststoff": "Plastic/silicone", "Glas": "Glass",
                            "Gestein": "Other", "Holz": "Other", "Textil": "Fabric shade"}
# (c) When a product lists SEVERAL materials, the part they belong to ("Materialzusatz (...)") decides:
#     shade/diffuser = look-defining (score 3), no part given = 2, base/frame/handle = 1.
#     If scores tie: the photo model decides (if available), otherwise MATERIAL_PRIORITY (first = wins).
MATERIAL_PART_SHADE = ["lampenschirm", "schirm", "diffusor", "kugel", "aussen-material", "aussenmaterial"]
MATERIAL_PART_BASE  = ["gestell", "struktur", "griff", "fuss", "fusse", "halterung", "sockel", "basis", "riemen",
                       "stromkabel", "innen-material", "innenmaterial"]
MATERIAL_IGNORE_VALUES = ["magnet"]        # hidden helper materials, never decisive
MATERIAL_PRIORITY = ["Glass", "Ceramic/porcelain", "Fabric shade", "Metal", "Plastic/silicone", "Other"]

# Words used for material fields without a family (Shopify Material, product field Material, product text)
MATERIAL_KEYWORDS = {
    "Plastic/silicone": ["acrylglas", "plexiglas", "kunststoff", "silikon", "silicone", "plastic", "polycarbonat",
                         "acryl", "polyacryl", "polyethylen", "polypropylen", "polyvinylchlorid", "technopolymer",
                         "methacrylat", "pmma", "gummi", "rubber", "=abs", "=pc", "=pe", "=pp", "=pvc", "=pet", "=hdpe"],
    "Metal": ["edelstahl", "stahl", "aluminium", "aluminum", "alu", "metall", "metal", "eisen", "zink", "titan",
              "messing", "kupfer", "bronze", "stainless", "steel", "iron", "oxidiert", "brass", "copper"],
    "Glass": ["glas", "glass", "borosilikat", "opalglas"],
    "Ceramic/porcelain": ["keramik", "porzellan", "ceramic", "porcelain", "steingut", "steinzeug"],
    "Fabric shade": ["textil", "leinen", "linen", "fabric", "stoffschirm", "baumwoll", "cotton", "canvas", "stoff"],
    "Other": ["marmor", "marble", "stein", "stone", "holz", "wood", "beton", "concrete", "travertin", "breccia",
              "rattan", "bambus", "bamboo", "kork", "cork", "leder", "leather", "papier", "paper", "furnier",
              "eiche", "esche", "douglasie", "wenge"],
}
MATERIAL_FINISH_KEYWORDS = {   # colour/finish words that only hint at metal (Low confidence, product text only)
    "Metal": ["chrom", "chrome", "messing", "brass", "kupfer", "copper", "bronze", "nickel", "silber", "gold"],
}

# ---------- Photo: zero-shot prompts for the optional image model, per (category, attribute) ----------
MATERIAL_PROMPTS = {
    "Ceramic/porcelain": "a product photo of a ceramic or porcelain lamp",
    "Glass": "a product photo of a lamp made of glass",
    "Metal": "a product photo of a lamp made of metal",
    "Fabric shade": "a product photo of a lamp with a fabric shade",
    "Plastic/silicone": "a product photo of a lamp made of plastic or silicone",
    "Other": "a product photo of a lamp made of stone, wood or concrete",
}
IMAGE_PROMPTS = {("Akkuleuchten", "Material & look"): MATERIAL_PROMPTS}

# ---------- Light control ----------
# If several functions are supported, the more capable one wins (inside the same evidence level and confidence).
LIGHT_PRIORITY = ["Colour-changing (RGB)", "Smart (app or remote)", "Tunable white (warm to cool)", "Dimmable",
                  "Fixed warm white"]
LIGHT_ATTR = {"dimming": ["Dimmbar (Leuchten)", "Dimmbar (Leuchtmittel)"],
              "colour_temperature": ["Lichtfarbe, Farbtemperatur"],
              "remote": ["Fernbedienung enthalten"]}        # value True -> Smart (app or remote)
DIMMING_IGNORE_REGEX = r"on-off"                            # text that says nothing about dimming
DIMMING_NO_REGEX  = r"^(nein|no|false)\b|nicht dimm|undimm|non-dimm"
DIMMING_YES_REGEX = r"dimm|^(ja|yes|true)$|touch"
# Keywords in attribute values / product text (listed in priority order; '=' = whole word only)
LIGHT_KEYWORDS = {
    "Colour-changing (RGB)": ["=rgb", "rgbw", "farbwechsel", "colour change", "color change", "colour-changing",
                              "color-changing", "mehrfarbig", "multicolor"],
    "Smart (app or remote)": ["=app", "bluetooth", "wlan", "wifi", "=smart", "zigbee", "fernbedienung", "remote",
                              "homekit", "matter"],
    "Tunable white (warm to cool)": ["color tune", "colour tune", "tunable", "=cct", "farbtemperatur",
                                     "warm to cool", "kaltweiss"],
    "Dimmable": ["dimm", "touch-dimm", "helligkeitsstufen", "brightness", "stufenlos"],
    "Fixed warm white": ["nicht dimmbar", "non-dimmable", "undimmable"],
}
LIGHT_FIXED_DEFAULT = False   # True: no dimming info + single warm-white colour temperature -> 'Fixed warm white' (Low)

# ---------- Environment ----------
ENV_ATTR = {"ip": ["Schutzart", "IP-Schutzart"], "season": ["Saisonartikel"]}
# second IP digit (water protection) -> environment: 0 = none, 1-3 = drip/spray, 4+ = splash/jet/immersion
IP_SECOND_DIGIT_TO_ENV = {0: "indoor", 1: "covered", 2: "covered", 3: "covered", 4: "full", 5: "full", 6: "full",
                          7: "full", 8: "full", 9: "full"}
ENV_SEASON_OUTDOOR_REGEX = r"outdoor"                       # value of 'Saisonartikel' that hints at outdoor use
ENV_WEATHER_KEYWORDS = ["wetterfest", "witterungsbestaendig", "witterungsbestandig", "wasserdicht", "weatherproof",
                        "weather-proof", "waterproof", "wetterbestaendig", "wetterbestandig", "spritzwasser",
                        "all-weather"]
ENV_OUTDOOR_KEYWORDS = ["outdoor", "aussen", "garten", "garden", "terrasse", "balkon", "solar", "camping",
                        "picknick", "patio"]
ENV_OUTDOOR_MAIN_CATEGORIES = ["aussenleuchten"]     # product main category (ItemAttribute) hinting outdoor use
ENV_VALUES = {"indoor": "Indoor only", "covered": "Covered outdoor (balcony, terrace)",
              "full": "Full outdoor (weather-rated)"}

# Level 4: used ONLY when attributes, text and photo give nothing (None = leave as 'Not determinable'). Low confidence.
DEFAULT_WHEN_NO_EVIDENCE = {
    "Environment": "Indoor only",
    "Light control": None,
}

# Product type from the name (only used as a similarity feature)
PRODUCT_TYPE_KEYWORDS = {"table lamp": ["tischleuchte", "tischlampe"], "floor lamp": ["stehleuchte", "stehlampe"],
                         "wall lamp": ["wandleuchte", "wandlampe"], "pendant": ["pendel", "haenge", "hange"],
                         "lantern": ["laterne", "lantern"], "garden/outdoor": ["gartenleuchte", "wegeleuchte"]}

# ItemAttribute names used as evidence (first found wins; several values are all kept)
ATTR_ALIASES = {"series": ["Herstellerserie"], "shop_colour": ["Shopify Farbe"], "shop_material": ["Shopify Material"],
                "shop_main_category": ["Shopify Main Category"], "ip_code": ["Schutzart", "IP-Schutzart"]}
# Attributes ignored when building free text from ItemAttribute (IDs, registry numbers, dimming/colour-temperature, ...)
ATTR_TEXT_SKIP_REGEX = (r"nummer|code|registr|melderegister|delist|nachfolge|printkatalog|category|weee|battg|eprel"
                        r"|dimm|lichtfarbe|material|saisonartikel|schutz|hinweis|verpackung|auslieferung|fernbedienung")

In [ ]:
# =====================================================================
# 1. SETUP, LOGGING AND HELPER FUNCTIONS
# =====================================================================
import re, time, hashlib, unicodedata, warnings, colorsys
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
from IPython.display import display
warnings.filterwarnings("ignore")
pd.set_option("display.width", 170); pd.set_option("display.max_colwidth", 50); pd.set_option("display.max_rows", 25)

import requests                                   # image download
from PIL import Image                             # image validation / features
from sklearn.feature_extraction.text import TfidfVectorizer   # text similarity

for folder in (RES_DIR, CACHE_DIR, INTERIM_DIR):
    folder.mkdir(parents=True, exist_ok=True)
if not SRC_DIR.exists():
    raise FileNotFoundError(f"Source folder not found: {SRC_DIR}  (set PROJECT_DIR in the configuration)")

# ---- compact logging: everything goes to the log list, only key lines are printed ----
LOG, ISSUES, COUNTS = [], [], {}
def log(msg, level="INFO", show=False):
    LOG.append(f"{time.strftime('%H:%M:%S')} [{level}] {msg}")
    if show: print(msg)
def issue(topic, count, detail=""):
    if count:
        ISSUES.append({"topic": topic, "count": int(count), "detail": detail})
        log(f"{topic}: {int(count)} {detail}", "WARN")

# ---- text normalisation ----
MISSING_TOKENS = {"", "0", "-", "--", "nan", "none", "null", "n/a", "na", "#n/a"}
def clean_text(x):
    # trim, collapse spaces, turn placeholders ('0', '-', 'null' ...) into None
    if x is None or (isinstance(x, float) and np.isnan(x)): return None
    s = re.sub(r"\s+", " ", str(x).replace("\u00a0", " ")).strip()
    return None if s.lower() in MISSING_TOKENS else s
def fold(s):
    # lower case, no accents, sharp s -> ss  ("Grün" -> "grun", "weiß" -> "weiss")
    s = unicodedata.normalize("NFKD", str(s).lower().replace("ß", "ss"))
    return "".join(c for c in s if not unicodedata.combining(c))
def norm_name(s):  return re.sub(r"[^a-z0-9]", "", fold(s))          # for file / column matching
def norm_label(s): return re.sub(r"\s+", " ", fold(s)).strip()        # for category / value matching
def norm_key(x):
    if x is None or (isinstance(x, float) and np.isnan(x)): return None
    s = re.sub(r"\.0+$", "", str(x).strip())
    return s or None
def parse_number(x):
    # handles '1.234', '1.234,5', '2,5', '-12', '' -> NaN
    s = str(x).strip().replace(" ", "")
    if s in ("", "-", "nan"): return np.nan
    if "," in s and "." in s:   s = s.replace(".", "").replace(",", ".") if s.rfind(",") > s.rfind(".") else s.replace(",", "")
    elif "," in s:              s = s.replace(",", ".")
    elif re.fullmatch(r"-?\d{1,3}(\.\d{3})+", s): s = s.replace(".", "")
    try: return float(s)
    except ValueError: return np.nan

# ---- safe file reading ----
def find_file(pattern, folder=None):
    folder = folder or SRC_DIR
    pat = norm_name(pattern)
    hits = [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in (".csv", ".txt", ".tsv", ".xlsx", ".xls")
            and pat in norm_name(p.stem)]
    if not hits:                      # tolerate small typos in file names (e.g. 'DesigStyle')
        import difflib
        files = [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in (".csv", ".txt", ".tsv", ".xlsx", ".xls")]
        close = difflib.get_close_matches(pat, [norm_name(p.stem) for p in files], n=1, cutoff=0.8)
        hits = [p for p in files if close and norm_name(p.stem) == close[0]]
    return sorted(hits, key=lambda p: len(p.name))[0] if hits else None
def sniff_sep(path):
    with open(path, "rb") as f: line = f.readline().decode("latin-1")
    line = re.sub(r'"[^"]*"', "", line)
    counts = {s: line.count(s) for s in [",", ";", "\t", "|"]}
    return max(counts, key=counts.get) if max(counts.values()) > 0 else ","
def read_table(path):
    # all columns as text; tries several encodings; detects the delimiter
    path = Path(path)
    if path.suffix.lower() in (".xlsx", ".xls"):
        df, meta = pd.read_excel(path, dtype=str).fillna(""), {"sep": "excel", "encoding": "excel"}
    else:
        sep, df = sniff_sep(path), None
        for enc in ("utf-8-sig", "cp1252", "latin-1"):
            try:
                df = pd.read_csv(path, sep=sep, encoding=enc, dtype=str, keep_default_na=False, na_filter=False,
                                 on_bad_lines="skip")
                meta = {"sep": sep, "encoding": enc}; break
            except UnicodeDecodeError:
                continue
        if df is None: raise ValueError(f"Could not decode {path.name}")
    df.columns = [re.sub(r"\s+", " ", str(c).replace("\ufeff", "")).strip() for c in df.columns]
    return df, meta
def resolve_columns(df, spec, required, table):
    lookup = {norm_name(c): c for c in df.columns}
    found = {canon: next((lookup[norm_name(c)] for c in cands if norm_name(c) in lookup), None) for canon, cands in spec.items()}
    missing = [k for k in required if not found.get(k)]
    if missing:
        raise KeyError(f"[{table}] required column(s) {missing} not found. Available: {list(df.columns)}. Adjust COLUMN_MAP.")
    return found

# ---- confidence levels (see README) ----
LEVELS = ["Not determinable", "Low", "Medium", "High"]
def shift_level(level, n): return LEVELS[max(1, min(3, LEVELS.index(level) + n))]
def result(value, conf, evidence, source):
    return {"value": value, "confidence": conf, "evidence": evidence, "source": source}
def undetermined(reason):
    return result(NOT_DETERMINABLE, "Not determinable", reason, "No reliable evidence")

# ---- keyword matching ----
def find_keywords(text, mapping):
    # returns [(position, value, keyword)] sorted by position; longest keywords first, matched text is consumed
    t = " " + fold(text or "") + " "
    flat = sorted(((fold(kw.lstrip("=")), kw.startswith("="), val) for val, kws in mapping.items() for kw in kws),
                  key=lambda x: -len(x[0]))
    hits = []
    for kw, whole, val in flat:
        pattern = r"(?<![a-z0-9])" + re.escape(kw) + (r"(?![a-z0-9])" if whole else "")
        for m in list(re.finditer(pattern, t)):
            if t[m.start():m.end()].strip() == "": continue
            hits.append((m.start(), val, kw)); t = t[:m.start()] + " " * (m.end() - m.start()) + t[m.end():]
    return sorted(hits)

COLOUR_TOKEN_MAP = {fold(kw): grp for grp, kws in COLOUR_KEYWORDS.items() for kw in kws}
def colour_groups(text):
    # colour groups named in a text (token based; handles compounds like 'cremeweiss', 'hellblau')
    groups = []
    for tok in re.findall(r"[a-z]+", fold(text or "")):
        grp = COLOUR_TOKEN_MAP.get(tok)
        if grp is None:
            base = next((b for b in sorted(COLOUR_COMPOUND_BASES, key=len, reverse=True) if tok.endswith(b) and len(tok) > len(b) + 2), None)
            grp = COLOUR_TOKEN_MAP.get(base) if base else None
        if grp and grp not in groups: groups.append(grp)
    return groups

log("Setup complete")

In [ ]:
# =====================================================================
# 2. LOAD AND VALIDATE SOURCE FILES
# =====================================================================
REQUIRED_COLS = {"products": ["item_id", "name", "subcategory"], "media": ["item_id", "url"],
                 "attributes": ["item_id", "name", "value"], "allowed": ["category", "attribute", "value"],
                 "sales": ["item_id"]}
OPTIONAL_TABLES = {"media", "attributes", "sales"}      # the workflow continues without these (with a warning)

TABLES, schema_rows = {}, []
for tname, pattern in FILE_PATTERNS.items():
    path = find_file(pattern)
    if path is None:
        if tname in OPTIONAL_TABLES:
            issue(f"optional file missing: {tname}", 1, f"pattern '{pattern}' in {SRC_DIR.name}"); continue
        raise FileNotFoundError(f"Required file '{pattern}' not found in {SRC_DIR}")
    df, meta = read_table(path)
    try:
        cols = resolve_columns(df, COLUMN_MAP[tname], REQUIRED_COLS[tname], tname)
    except KeyError as e:
        if tname in OPTIONAL_TABLES: issue(f"optional file unusable: {tname}", 1, str(e)); continue
        raise
    TABLES[tname] = {"df": df, "path": path, "meta": meta, "cols": cols}
    COUNTS[f"{tname}_rows_read"] = len(df)
    schema_rows.append({"table": tname, "file": path.name, "rows": len(df), "delimiter": repr(meta["sep"]),
                        "encoding": meta["encoding"], "mapped": f"{sum(v is not None for v in cols.values())}/{len(cols)}",
                        "unmapped optional": ", ".join(k for k, v in cols.items() if v is None) or "-"})
    log(f"{tname}: {path.name}, {len(df)} rows, columns {cols}")
display(pd.DataFrame(schema_rows))

In [ ]:
# =====================================================================
# 3. ALLOWED ATTRIBUTES AND VALUES  (+ 4. SELECT SUB-CATEGORIES, labels)
# =====================================================================
A = TABLES["allowed"]
allowed_df = pd.DataFrame({k: A["df"][v].map(clean_text) for k, v in A["cols"].items() if v}).dropna()
allowed_df["_k"] = allowed_df.apply(lambda r: "|".join(norm_label(r[c]) for c in ("category", "attribute", "value")), axis=1)
issue("duplicate rows in allowed attribute/value list (removed)", allowed_df.duplicated("_k").sum())
allowed_df = allowed_df.drop_duplicates("_k").drop(columns="_k")

def allowed_for(category_name):
    # {attribute: [values]} for one category; accepts partial names ('Esstisch' -> 'Design-Esstisch')
    key, cats = norm_name(category_name), allowed_df["category"].unique()
    match = [c for c in cats if norm_name(c) == key] or [c for c in cats if key in norm_name(c)]
    if not match: raise KeyError(f"Category '{category_name}' not found in allowed list. Available: {list(cats)}")
    sub = allowed_df[allowed_df["category"] == match[0]]
    return {attr: g["value"].tolist() for attr, g in sub.groupby("attribute", sort=False)}
ALLOWED = {cat: allowed_for(cfg["allowed_category"]) for cat, cfg in CATEGORIES.items()}

def canonical_value(value, allowed_values):
    # returns the allowed spelling of a value, or None if it is not allowed
    lookup = {norm_label(v): v for v in allowed_values}
    return lookup.get(norm_label(value)) if value is not None else None

# ---- products: normalise ----
P_T = TABLES["products"]
products_raw = pd.DataFrame({k: P_T["df"][v] for k, v in P_T["cols"].items() if v})
for c in products_raw.columns:
    products_raw[c] = products_raw[c].map(norm_key if c in ("item_id", "manufacturer_id") else clean_text)
for c in ("length", "depth", "height", "weight"):
    if c in products_raw: products_raw[c] = pd.to_numeric(products_raw[c], errors="coerce").replace(0, np.nan)  # 0 = unknown
issue("products without item id (dropped)", products_raw["item_id"].isna().sum())
products_raw = products_raw.dropna(subset=["item_id"])
issue("exact duplicate product rows (removed)", products_raw.duplicated().sum())
products_raw = products_raw.drop_duplicates()
issue("duplicate product ids with different content (first kept)", products_raw.duplicated("item_id").sum())
products_raw = products_raw.drop_duplicates("item_id").set_index("item_id", drop=False)
products_raw.index.name = None
COUNTS["products_unique"] = len(products_raw)

# ---- select the configured sub-categories ----
products_raw["_subcat_norm"] = products_raw["subcategory"].map(lambda s: norm_label(s) if s else "")
selected_parts = []
for cat, cfg in CATEGORIES.items():
    wanted = {norm_label(v) for v in cfg["dwh_subcategory"]}
    part = products_raw[products_raw["_subcat_norm"].isin(wanted)].copy()
    part["category"] = cat
    if part.empty:
        close = products_raw["subcategory"].dropna().unique()
        issue(f"no products found for {cat}", 1, "similar names: " + ", ".join(s for s in close if norm_label(cfg['allowed_category'])[:4] in norm_label(s))[:200])
    selected_parts.append(part)
selected = pd.concat(selected_parts)
issue("product ids selected for more than one category", selected["item_id"].duplicated().sum())
COUNTS["products_selected"] = len(selected)
for cat in CATEGORIES: COUNTS[f"selected_{cat}"] = int((selected["category"] == cat).sum())

# ---- specialist label files (one per category) ----
LABELS = {}     # category -> DataFrame(item_id, attribute, value) after validation
for cat, cfg in CATEGORIES.items():
    LABELS[cat] = pd.DataFrame(columns=["item_id", "attribute", "value"])
    if not cfg.get("label_file"): continue
    path = find_file(cfg["label_file"])
    if path is None:
        issue(f"label file not found for {cat}", 1, f"pattern '{cfg['label_file']}' - design style cannot be transferred"); continue
    ldf, _ = read_table(path)
    lcols = resolve_columns(ldf, COLUMN_MAP["labels"], ["item_id", "attribute", "value"], f"labels/{cat}")
    lab = pd.DataFrame({"item_id": ldf[lcols["item_id"]].map(norm_key), "attribute": ldf[lcols["attribute"]].map(clean_text),
                        "value": ldf[lcols["value"]].map(clean_text)}).dropna()
    # map attribute + value onto the allowed spelling; drop what is not allowed
    attr_lookup = {norm_label(a): a for a in ALLOWED[cat]}
    lab["attribute"] = lab["attribute"].map(lambda a: attr_lookup.get(norm_label(a)))
    bad_attr = lab["attribute"].isna()
    issue(f"label rows with an attribute not defined for {cat} (ignored)", bad_attr.sum())
    lab = lab[~bad_attr].copy()
    lab["value"] = [canonical_value(v, ALLOWED[cat][a]) for a, v in zip(lab["attribute"], lab["value"])]
    issue(f"label rows with a value not allowed for {cat} (ignored)", lab["value"].isna().sum())
    lab = lab.dropna()
    conflicts = lab.groupby(["item_id", "attribute"])["value"].nunique().gt(1).sum()
    issue(f"products with conflicting labels in {path.name} (first kept)", conflicts)
    issue(f"duplicate label rows in {path.name} (removed)", lab.duplicated(["item_id", "attribute"]).sum() - conflicts)
    LABELS[cat] = lab.drop_duplicates(["item_id", "attribute"]).reset_index(drop=True)
    in_dwh = LABELS[cat]["item_id"].isin(products_raw.index)
    in_cat = LABELS[cat]["item_id"].isin(selected.loc[selected["category"] == cat, "item_id"])
    issue(f"labelled products not found in product file ({cat})", (~in_dwh).sum(), "ids: " + ", ".join(LABELS[cat].loc[~in_dwh, 'item_id'].unique()[:12]))
    COUNTS[f"labels_{cat}"] = len(LABELS[cat]); COUNTS[f"labels_{cat}_in_category"] = int(in_cat.sum())
    COUNTS[f"labels_{cat}_other_subcategory"] = int((in_dwh & ~in_cat).sum())

# ---- reference-only products (labelled, but outside the selected sub-category) ----
ref_parts = []
for cat in CATEGORIES:
    if not REFERENCE_FROM_ANY_SUBCATEGORY: continue
    ids = set(LABELS[cat]["item_id"]) - set(selected.loc[selected["category"] == cat, "item_id"])
    ref = products_raw[products_raw.index.isin(ids)].copy(); ref["category"] = cat
    ref_parts.append(ref)
selected["is_target"] = True
references = pd.concat(ref_parts) if ref_parts else selected.iloc[0:0].copy()
references["is_target"] = False
PRODUCTS = pd.concat([selected, references])
PRODUCTS["item_cat"] = PRODUCTS["item_id"] + "|" + PRODUCTS["category"]
PRODUCTS = PRODUCTS.drop_duplicates("item_cat").set_index("item_cat", drop=False)
PRODUCTS.index.name = None

summary = selected.groupby("category").size().rename("products selected").to_frame()
summary["labelled (in category)"] = [COUNTS.get(f"labels_{c}_in_category", 0) for c in summary.index]
summary["allowed attributes"] = [len(ALLOWED[c]) for c in summary.index]
display(summary)

In [ ]:
# =====================================================================
# 5. JOIN PRODUCT DATA, EXISTING ATTRIBUTES AND IMAGE LINKS
# =====================================================================
wanted_ids = set(PRODUCTS["item_id"])

# ---- 5.1 existing product attributes (ItemAttribute.csv) ----
# One product can have the same attribute name several times (e.g. 'Material (Kunststoff)' in two attribute sets),
# so ALL distinct values are kept:  ATTR[item][attribute name (normalised)] = [value, value, ...]
ATTR, ATTR_SETS = {}, {}      # item -> {name: [values]} / item -> set of attribute-set names
if "attributes" in TABLES:
    T = TABLES["attributes"]
    at = pd.DataFrame({k: T["df"][v] for k, v in T["cols"].items() if v})
    at["item_id"] = at["item_id"].map(norm_key)
    at = at[at["item_id"].isin(wanted_ids)].copy()
    for c in ("name", "value") + (("set",) if "set" in at else ()): at[c] = at[c].map(clean_text)   # 'Null', '0', '-' -> missing
    n_missing = int(at["value"].isna().sum())
    at = at.dropna(subset=["name", "value"])
    at["_name"] = at["name"].map(norm_label)
    at["_set"] = at["set"].map(lambda x: norm_label(x) if x else "") if "set" in at else ""
    issue("attribute rows with empty value / 'Null' (ignored)", n_missing)
    issue("exact duplicate attribute rows (removed)", at.duplicated(["item_id", "_set", "_name", "value"]).sum())
    at = at.drop_duplicates(["item_id", "_set", "_name", "value"])
    at_u = at.drop_duplicates(["item_id", "_name", "value"])      # same name+value in two sets counts once
    issue("products with the same attribute name but different values (all values kept)",
          at_u.duplicated(["item_id", "_name"]).sum())
    for item, g in at_u.groupby("item_id"):
        d = {}
        for n_, v_ in zip(g["_name"], g["value"]): d.setdefault(n_, []).append(v_)
        ATTR[item] = d
    ATTR_SETS = at[at["_set"] != ""].groupby("item_id")["_set"].agg(set).to_dict()
    COUNTS["attribute_rows_used"] = len(at_u)

def get_attr_all(item, aliases):
    # all distinct values of the first alias(es) that exist for this product
    out = []
    for a in aliases:
        for v in ATTR.get(item, {}).get(norm_label(a), []):
            if v not in out: out.append(v)
    return out
def get_attr(item, aliases):
    vals = get_attr_all(item, aliases)
    return vals[0] if vals else None
def attribute_free_text(item):
    # text-like attribute values (without ids, registry numbers, dimming, ...) for keyword search
    parts = [f"{n} {v}" for n, vals in ATTR.get(item, {}).items() if not re.search(ATTR_TEXT_SKIP_REGEX, n)
             for v in vals if re.search(r"[A-Za-zÄÖÜäöüß]{3}", v)]
    return " | ".join(parts)

# ---- 5.2 derived product fields ----
def split_manufacturer(name):
    parts = re.split(r"\s+[-–]\s+", name or "", maxsplit=1)
    return (parts[0].strip(), parts[1].strip()) if len(parts) == 2 else (None, name or "")
def product_type_from_name(name):
    hits = find_keywords(name, PRODUCT_TYPE_KEYWORDS)
    return hits[0][1] if hits else None

PRODUCTS["manufacturer_name"] = PRODUCTS["name"].map(lambda n: split_manufacturer(n)[0])
PRODUCTS["name_short"] = PRODUCTS["name"].map(lambda n: split_manufacturer(n)[1])
PRODUCTS["name_tail"] = PRODUCTS["name_short"].map(lambda n: n.split(",")[-1].strip() if n else n)
PRODUCTS["product_type"] = PRODUCTS["name"].map(product_type_from_name)
for col, aliases in ATTR_ALIASES.items():
    PRODUCTS[col] = PRODUCTS["item_id"].map(lambda i, a=aliases: get_attr(i, a))
PRODUCTS["shop_main_category"] = PRODUCTS["shop_main_category"].map(lambda s: norm_label(s) if s else None)
PRODUCTS["attr_text"] = PRODUCTS["item_id"].map(attribute_free_text)
PRODUCTS["attr_sets"] = PRODUCTS["item_id"].map(lambda i: ATTR_SETS.get(i, set()))
PRODUCTS["n_attributes"] = PRODUCTS["item_id"].map(lambda i: len(ATTR.get(i, {})))

# ---- 5.3 optional extra information (tier, price, ...) ----
EXTRA_ROLES = {}
for spec in EXTRA_PRODUCT_FILES:
    path = find_file(spec["file_pattern"])
    if path is None: issue("extra product file not found", 1, spec["file_pattern"]); continue
    edf, _ = read_table(path)
    lookup = {norm_name(c): c for c in edf.columns}
    key_col = next((lookup[norm_name(k)] for k in spec["key"] if norm_name(k) in lookup), None)
    if key_col is None: issue("extra file without key column", 1, path.name); continue
    keep = {}
    for col, role in spec["columns"].items():
        if norm_name(col) in lookup: keep[col] = lookup[norm_name(col)]; EXTRA_ROLES[col] = role
        else: issue("extra column not found", 1, f"{path.name}: {col}")
    ex = pd.DataFrame({"item_id": edf[key_col].map(norm_key), **{c: edf[v] for c, v in keep.items()}}).dropna(subset=["item_id"])
    issue(f"duplicate keys in {path.name} (first kept)", ex.duplicated("item_id").sum())
    ex = ex.drop_duplicates("item_id").set_index("item_id")
    for col, role in ((c, EXTRA_ROLES[c]) for c in keep):
        PRODUCTS[col] = PRODUCTS["item_id"].map(ex[col])
        PRODUCTS[col] = pd.to_numeric(PRODUCTS[col].map(parse_number), errors="coerce") if role == "numeric" else PRODUCTS[col].map(clean_text)
for col, spec in EXTRA_ATTRIBUTE_FEATURES.items():
    EXTRA_ROLES[col] = spec["role"]
    vals = PRODUCTS["item_id"].map(lambda i, s=spec: get_attr(i, [s["attribute"]]))
    PRODUCTS[col] = vals.map(parse_number) if spec["role"] == "numeric" else vals
if EXTRA_ROLES: log(f"extra product columns: {EXTRA_ROLES}")

# ---- 5.4 image links (ItemMedia.csv) ----
PRODUCTS["image_urls"] = [[] for _ in range(len(PRODUCTS))]
if "media" in TABLES:
    T = TABLES["media"]
    md_ = pd.DataFrame({k: T["df"][v] for k, v in T["cols"].items() if v})
    md_["item_id"] = md_["item_id"].map(norm_key); md_["url"] = md_["url"].map(clean_text)
    md_ = md_[md_["item_id"].isin(wanted_ids)].dropna(subset=["url"])
    issue("duplicate image links (removed)", md_.duplicated(["item_id", "url"]).sum())
    md_ = md_.drop_duplicates(["item_id", "url"])
    md_["_order"] = pd.to_numeric(md_.get("order", pd.Series(1, index=md_.index)), errors="coerce").fillna(10**6)
    md_ = md_[~md_["_order"].isin(IMAGE_SETTINGS["skip_order_values"])]
    md_["url"] = md_["url"].map(lambda u: u if re.match(r"https?://", u) else (IMAGE_SETTINGS["base_url"] + u if IMAGE_SETTINGS["base_url"] else None))
    issue("image links that are not valid URLs (ignored)", md_["url"].isna().sum())
    md_ = md_.dropna(subset=["url"]).sort_values(["item_id", "_order"])
    urls = md_.groupby("item_id")["url"].agg(lambda s: list(s)[:IMAGE_SETTINGS["max_images_per_product"]]).to_dict()
    PRODUCTS["image_urls"] = PRODUCTS["item_id"].map(lambda i: urls.get(i, []))

tgt = PRODUCTS[PRODUCTS["is_target"]]
join_report = pd.DataFrame({"target products": [len(tgt)],
                            "without image link": [int((tgt["image_urls"].map(len) == 0).sum())],
                            "without any ItemAttribute": [int((tgt["n_attributes"] == 0).sum())],
                            "without manufacturer in name": [int(tgt["manufacturer_name"].isna().sum())],
                            "without dimensions": [int(tgt[["length", "depth", "height"]].isna().any(axis=1).sum())]})
for c in join_report.columns[1:]: issue(f"target products {c}", join_report[c].iloc[0])
display(join_report)

In [ ]:
# =====================================================================
# 6. DOWNLOAD, CACHE AND VALIDATE PRODUCT IMAGES
# =====================================================================
S = IMAGE_SETTINGS
def cache_file(item_id, url):
    return CACHE_DIR / f"{item_id}_{hashlib.md5(url.encode()).hexdigest()[:10]}.jpg"

def fetch_image(job):
    # downloads one image, validates it with PIL and stores a resized JPEG; never raises
    item_id, url = job
    dest = cache_file(item_id, url)
    if dest.exists() and dest.stat().st_size > 0:
        return {"item_id": item_id, "url": url, "status": "cached", "path": str(dest), "reason": ""}
    if not S["download"]:
        return {"item_id": item_id, "url": url, "status": "skipped", "path": "", "reason": "download disabled"}
    reason = ""
    for attempt in range(S["retries"] + 1):
        try:
            r = requests.get(url, headers={"User-Agent": S["user_agent"]}, timeout=(S["connect_timeout"], S["read_timeout"]))
            if r.status_code != 200:
                reason = f"HTTP {r.status_code}"
                if r.status_code in (400, 401, 403, 404, 410): break
                continue
            img = Image.open(BytesIO(r.content)); img.verify()                 # is it a real image?
            img = Image.open(BytesIO(r.content)).convert("RGB")
            if min(img.size) < S["min_side"]: reason = f"image too small {img.size}"; break
            img.thumbnail((S["max_side"], S["max_side"])); img.save(dest, "JPEG", quality=90)
            return {"item_id": item_id, "url": url, "status": "downloaded", "path": str(dest), "reason": ""}
        except Exception as e:                                                   # network, decoding, ...
            reason = type(e).__name__
            time.sleep(0.5 * (attempt + 1))
    return {"item_id": item_id, "url": url, "status": "failed", "path": "", "reason": reason}

jobs = [(r.item_id, u) for r in PRODUCTS.itertuples() for u in r.image_urls]
jobs = list(dict.fromkeys(jobs))
image_log, aborted = [], False
with ThreadPoolExecutor(max_workers=S["workers"]) as pool:
    batch = max(S["abort_after_failures"], 20)
    for start in range(0, len(jobs), batch):
        image_log += list(pool.map(fetch_image, jobs[start:start + batch]))
        ok_so_far = any(x["status"] in ("downloaded", "cached") for x in image_log)
        if S["abort_after_failures"] and len(image_log) >= S["abort_after_failures"] and not ok_so_far:
            aborted = True; break
for job in jobs[len(image_log):]:
    image_log.append({"item_id": job[0], "url": job[1], "status": "skipped", "path": "", "reason": "aborted after repeated failures"})
image_log = pd.DataFrame(image_log, columns=["item_id", "url", "status", "path", "reason"])
image_log.to_csv(INTERIM_DIR / "image_download_log.csv", index=False, sep=CSV_SEP, encoding=CSV_ENCODING)

# main image per product = first usable image
usable = image_log[image_log["status"].isin(["downloaded", "cached"])].drop_duplicates("item_id").set_index("item_id")["path"]
PRODUCTS["image_path"] = PRODUCTS["item_id"].map(usable)
def image_status(row):
    if not row["image_urls"]: return "no image link"
    if isinstance(row["image_path"], str): return "ok"
    fails = image_log[(image_log["item_id"] == row["item_id"]) & (image_log["reason"] != "")]["reason"]
    return "failed: " + (fails.iloc[0] if len(fails) else "not downloaded")
PRODUCTS["image_status"] = PRODUCTS.apply(image_status, axis=1)
counts = image_log["status"].value_counts().to_dict()
COUNTS["image_files_" + "_".join(f"{k}{v}" for k, v in sorted(counts.items()))] = len(image_log)
issue("failed image downloads", counts.get("failed", 0), "reasons: " + str(image_log.loc[image_log.status == 'failed', 'reason'].value_counts().to_dict()))
if aborted: issue("image download aborted early", 1, "first downloads all failed - check internet/proxy, then re-run (cached images are kept)")
n_tgt = int(PRODUCTS["is_target"].sum()); n_ok = int(PRODUCTS.loc[PRODUCTS["is_target"], "image_path"].notna().sum())
log(f"Images: {n_ok}/{n_tgt} target products with a usable image {counts}", show=True)

In [ ]:
# =====================================================================
# 7. IMAGE FEATURES (shape / colour; optional CLIP model)
# =====================================================================
def extract_image_features(path):
    # shape descriptor of the product silhouette + dominant colour of the product (background removed)
    img = Image.open(path).convert("RGB"); img.thumbnail((160, 160))
    arr = np.asarray(img).astype(float)
    border = np.concatenate([arr[0], arr[-1], arr[:, 0], arr[:, -1]])
    mask = np.linalg.norm(arr - np.median(border, axis=0), axis=2) > IMAGE_FEATURES["bg_tolerance"]
    h_, w_ = mask.shape
    if not (0.01 < mask.mean() < 0.95):                                  # background could not be separated
        centre = arr[h_ // 4: 3 * h_ // 4, w_ // 4: 3 * w_ // 4].reshape(-1, 3)
        return {"shape": None, "rgb": np.median(centre, axis=0), "fg_share": np.nan}
    ys, xs = np.where(mask); y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    crop = mask[y0:y1, x0:x1]; h, w = crop.shape; side = max(h, w)
    square = np.zeros((side, side), bool); oy, ox = (side - h) // 2, (side - w) // 2; square[oy:oy + h, ox:ox + w] = crop
    sil = np.asarray(Image.fromarray((square * 255).astype("uint8")).resize((16, 16), Image.BILINEAR)) / 255.0
    gray = arr.mean(axis=2)[y0:y1, x0:x1]; gy, gx = np.gradient(gray)
    hist, _ = np.histogram(np.arctan2(gy, gx) % np.pi, bins=8, range=(0, np.pi), weights=np.hypot(gx, gy))
    shape = {"sil": sil.ravel(), "orient": hist / (hist.sum() + 1e-9), "geom": np.array([np.log(h / w), crop.mean()])}
    return {"shape": shape, "rgb": np.median(arr[mask], axis=0), "fg_share": float(mask.mean())}

IMG_FEAT = {}                    # item_id -> features
for item, path in PRODUCTS.drop_duplicates("item_id").set_index("item_id")["image_path"].dropna().items():
    try: IMG_FEAT[item] = extract_image_features(path)
    except Exception as e: issue("image feature extraction failed", 1, f"{item}: {type(e).__name__}")

# ---- OPTIONAL: CLIP image model (embeddings for similarity + zero-shot prompts). Needs the packages torch and transformers ----
CLIP = None
CLIP_EMB, CLIP_PROBS = {}, {}      # item -> embedding / (category, attribute) -> item -> {value: probability}
if USE_CLIP:
    try:
        import os; os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
        import torch
        import transformers; transformers.logging.set_verbosity_error()
        from transformers import CLIPModel, CLIPProcessor
        _model, _proc = CLIPModel.from_pretrained(CLIP_MODEL).eval(), CLIPProcessor.from_pretrained(CLIP_MODEL)
        def _tensor(x): return x if torch.is_tensor(x) else x.pooler_output
        def clip_images(paths, batch=16):
            out = []
            for i in range(0, len(paths), batch):
                imgs = [Image.open(p).convert("RGB") for p in paths[i:i + batch]]
                with torch.no_grad(): e = _tensor(_model.get_image_features(**_proc(images=imgs, return_tensors="pt")))
                out.append((e / e.norm(dim=-1, keepdim=True)).numpy())
            return np.vstack(out)
        def clip_texts(texts):
            with torch.no_grad(): e = _tensor(_model.get_text_features(**_proc(text=texts, return_tensors="pt", padding=True)))
            return (e / e.norm(dim=-1, keepdim=True)).numpy()
        paths = PRODUCTS.drop_duplicates("item_id").set_index("item_id")["image_path"].dropna()
        emb = clip_images(list(paths.values)); CLIP_EMB = dict(zip(paths.index, emb))
        for (cat_, attr_), prompts in IMAGE_PROMPTS.items():
            vals = list(prompts); tx = clip_texts([prompts[v] for v in vals])
            logits = 100 * emb @ tx.T; probs = np.exp(logits - logits.max(1, keepdims=True)); probs /= probs.sum(1, keepdims=True)
            CLIP_PROBS[(norm_label(cat_), norm_label(attr_))] = {i: dict(zip(vals, map(float, p))) for i, p in zip(paths.index, probs)}
        CLIP = True
    except Exception as e:
        issue("CLIP model not available - using shape/colour features only", 1, f"{type(e).__name__}: {str(e)[:120]}")
log(f"Image features for {len(IMG_FEAT)} products (mode: {'CLIP' if CLIP else 'shape + colour'})", show=True)

In [ ]:
# =====================================================================
# 8. SIMILARITY BETWEEN PRODUCTS  (text + structured data + image)
# =====================================================================
def unit_rows(M):
    return M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-9)
def as_text(x): return x if isinstance(x, str) else ""

def sim_text(frame):
    # TF-IDF character n-gram similarity of name (without colour words), series, manufacturer + extra text columns
    def build(r):
        tokens = [t for t in re.findall(r"[a-z]+", fold(as_text(r["name_short"]))) if t not in COLOUR_TOKEN_MAP]
        parts = [" ".join(tokens), fold(as_text(r["series"])), fold(as_text(r["series"])), fold(as_text(r["manufacturer_name"]))]
        parts += [fold(str(r[c])) for c, role in EXTRA_ROLES.items() if role == "text" and isinstance(r[c], str)]
        return " ".join(parts)
    X = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True).fit_transform(frame.apply(build, axis=1))
    return (X @ X.T).toarray()

def sim_structured(frame):
    # Gower-style similarity: numeric = 1 - scaled difference, categorical = equal (1) or not (0); missing values are skipped
    num = [c for c in NUMERIC_FEATURES if c in frame] + [c for c, r in EXTRA_ROLES.items() if r == "numeric"]
    cat = [c for c in CATEGORICAL_FEATURES if c in frame] + [c for c, r in EXTRA_ROLES.items() if r == "categorical"]
    n = len(frame); total, weight = np.zeros((n, n)), np.zeros((n, n))
    for c in num:
        v = np.log1p(np.clip(frame[c].astype(float).values, 0, None))
        if np.isnan(v).all(): continue
        lo, hi = np.nanpercentile(v, [5, 95]); span = max(hi - lo, 1e-6)
        ok = ~np.isnan(v); both = ok[:, None] & ok[None, :]
        d = np.abs(np.nan_to_num(v)[:, None] - np.nan_to_num(v)[None, :]) / span
        total += np.where(both, 1 - np.clip(d, 0, 1), 0); weight += both
    for c in cat:
        v = frame[c].astype(object).values; ok = np.array([isinstance(x, str) for x in v])
        both = ok[:, None] & ok[None, :]; eq = v[:, None] == v[None, :]
        total += np.where(both, eq.astype(float), 0); weight += both
    return np.where(weight > 0, total / np.maximum(weight, 1), np.nan)

def sim_image(frame):
    # correlation-type similarity of image descriptors (CLIP embedding, or silhouette + edge-direction + aspect)
    ids = frame["item_id"].tolist(); empty = np.full((len(ids), len(ids)), np.nan)
    if CLIP and CLIP_EMB:
        dim = len(next(iter(CLIP_EMB.values())))
        X = np.array([CLIP_EMB.get(i, np.full(dim, np.nan)) for i in ids])
    else:
        have = [i for i in ids if IMG_FEAT.get(i) and IMG_FEAT[i]["shape"] is not None]
        if len(have) < 2: return empty
        blocks = {k: np.array([IMG_FEAT[i]["shape"][k] for i in have]) for k in ("sil", "orient", "geom")}
        blocks = {k: (b - b.mean(0)) / (b.std(0) + 1e-6) if k == "geom" else b - b.mean(0) for k, b in blocks.items()}
        Z = np.hstack([unit_rows(blocks["sil"]), unit_rows(blocks["orient"]) * 0.5, unit_rows(blocks["geom"]) * 0.5])
        X = np.full((len(ids), Z.shape[1]), np.nan); X[[ids.index(i) for i in have]] = Z
    ok = ~np.isnan(X).any(axis=1)
    if ok.sum() >= 2:
        Zc = X[ok] - X[ok].mean(0); C = (unit_rows(Zc) @ unit_rows(Zc).T + 1) / 2
        empty[np.ix_(np.where(ok)[0], np.where(ok)[0])] = C
    return empty

SIM, SIM_PARTS = {}, {}        # category -> DataFrame (combined) / dict of component DataFrames
for cat in CATEGORIES:
    frame = PRODUCTS[PRODUCTS["category"] == cat].drop_duplicates("item_id").reset_index(drop=True)
    parts = {"text": sim_text(frame), "structured": sim_structured(frame), "image": sim_image(frame)}
    num = np.zeros_like(parts["text"]); den = np.zeros_like(parts["text"])
    for comp, w in SIMILARITY["weights"].items():
        M = parts[comp]; ok = ~np.isnan(M); num += np.where(ok, M * w, 0); den += np.where(ok, w, 0)
    ids = frame["item_id"].tolist()
    SIM[cat] = pd.DataFrame(np.where(den > 0, num / np.maximum(den, 1e-9), np.nan), index=ids, columns=ids)
    SIM_PARTS[cat] = {k: pd.DataFrame(v, index=ids, columns=ids) for k, v in parts.items()}
    n_img = int((~np.isnan(parts["image"]).all(axis=1)).sum())
    log(f"{cat}: similarity matrix {SIM[cat].shape}; image component for {n_img} products")
log("Similarity matrices computed (weights: " + str(SIMILARITY["weights"]) + ")", show=True)

In [ ]:
# =====================================================================
# 9. ASSIGNMENT RULES  (priority: 1 attributes -> 2 product text -> 3 photo -> 4 default)
# =====================================================================
# Every rule returns {value, confidence, evidence, source}. A rule builds a list of candidates with cand(); resolve() then
#   * uses ONLY the highest evidence level that has a candidate (attributes beat text, text beats photo),
#   * inside that level takes the highest confidence (list order = tie-break),
#   * raises the confidence by one level if a lower level independently confirms the same value.
def tier_of(source):
    # evidence level from the source label: 1 attribute, 2 text, 3 photo, 4 anything else
    return 1 if source.startswith("Product attribute") else 2 if source.startswith("Product text") else 3 if source.startswith("Image") else 4

def cand(value, level, note, source):
    return {"value": value, "level": level, "note": note, "source": source}

def resolve(cands, none_reason):
    if not cands: return undetermined(none_reason)
    top_tier = min(tier_of(c["source"]) for c in cands)
    top = [c for c in cands if tier_of(c["source"]) == top_tier]
    best_idx = max(LEVELS.index(c["level"]) for c in top)
    best = next(c for c in top if LEVELS.index(c["level"]) == best_idx)           # first in list wins a tie
    same = sorted([c for c in cands if c["value"] == best["value"]], key=lambda c: tier_of(c["source"]))
    sources = list(dict.fromkeys(c["source"] for c in same))
    agree = len({tier_of(x) for x in sources}) >= 2                               # confirmed by another evidence level
    level = LEVELS[min(3, best_idx + (1 if agree else 0))]
    notes = list(dict.fromkeys(c["note"] for c in same))
    others = [f"{c['value']} ({c['note']})" for c in cands if c["value"] != best["value"]]
    evidence = "; ".join(notes[:2]) + (f" (+{len(notes) - 2} more)" if len(notes) > 2 else "")
    if others: evidence += ". Not used (lower priority or weaker): " + "; ".join(others[:3])
    return result(best["value"], level, evidence, " + ".join(sources[:2]))

def as_bool(v): return fold(v).strip() in ("true", "ja", "yes", "1")

# ---------- Colour Group ----------
def rgb_to_group(rgb):
    # rough mapping of the dominant product colour to the colour groups (heuristic, hence Low confidence)
    r, g, b = [x / 255 for x in rgb]; h, s, v = colorsys.rgb_to_hsv(r, g, b); h *= 360
    if v < 0.28: return "Deep Shadows & Graphites"
    if s < 0.12: return "Light Neutrals" if v > 0.82 else ("Cool Grays & Minerals" if v > 0.45 else "Deep Shadows & Graphites")
    if s < 0.30 and 20 <= h <= 60: return "Light Neutrals" if v > 0.75 else "Warm Earth & Natural Woods"
    if 15 <= h <= 50 and s < 0.70 and v < 0.80: return "Warm Earth & Natural Woods"
    return "Chromatic Accents & Jewel Tones"

def assign_colour_group(rec, stage="evidence"):
    if stage != "evidence": return None
    cands = []
    for field, label in (("shop_colour", "Shopify colour"), ("colour", "colour name"), ("variant", "variant")):   # level 1
        text = rec.get(field)
        groups = colour_groups(text) if isinstance(text, str) else []
        if len(groups) == 1: cands.append(cand(groups[0], "High", f"{label} '{text}' -> {groups[0]}", "Product attribute"))
        elif len(groups) > 1: cands.append(cand(COLOUR_MIX_VALUE, "Medium", f"{label} '{text}' names several colour groups", "Product attribute"))
    groups = colour_groups(rec.get("name_tail"))                                                                  # level 2
    if len(groups) == 1: cands.append(cand(groups[0], "Medium", f"product name '{rec['name_tail']}' -> {groups[0]}", "Product text"))
    elif len(groups) > 1: cands.append(cand(COLOUR_MIX_VALUE, "Low", f"product name '{rec['name_tail']}' names several colour groups", "Product text"))
    feat = IMG_FEAT.get(rec["item_id"])                                                                           # level 3
    if feat is not None:
        g = rgb_to_group(feat["rgb"]); cands.append(cand(g, "Low", "dominant colour of the product photo", "Image"))
    return resolve(cands, "no colour word in attributes/name and no usable photo")

# ---------- Material & look ----------
def material_attribute_candidates(item, rec):
    # {allowed value: {"score": part score, "notes": [...]}} from ItemAttribute (+ structured product fields)
    names, found = ATTR.get(item, {}), {}
    def add(value, score, note, shown=None):
        e = found.setdefault(value, {"score": 0, "notes": [], "shown": []}); e["score"] = max(e["score"], score); e["notes"].append(note)
        if shown and shown not in e["shown"]: e["shown"].append(shown)
    parts = {}                                                      # family -> values of 'Materialzusatz (family)'
    for n, vals in names.items():
        m = re.match(r"materialzusatz\s*\d*\s*\(([^)]+)\)", n)
        if m: parts.setdefault(m.group(1).strip(), []).extend(vals)
    def part_info(family):
        words = [fold(v) for v in parts.get(family, [])]
        if any(w_ in x for x in words for w_ in MATERIAL_PART_SHADE): return 3, "shade/diffuser"
        if words and all(any(w_ in x for w_ in MATERIAL_PART_BASE) for x in words): return 1, "base/frame"
        return 2, ""
    for n, vals in names.items():                                   # (a) 'Material (Metall)', 'Material 2 (Kunststoff)', ...
        m = re.match(r"material\s*\d*\s*\(([^)]+)\)", n)
        if not m: continue
        family = m.group(1).strip()
        for v in vals:
            if any(ig in fold(v) for ig in MATERIAL_IGNORE_VALUES): continue
            value = MATERIAL_ATTRIBUTE_FAMILIES.get(family)
            if value is None:
                hits = find_keywords(v, MATERIAL_KEYWORDS)
                value = hits[0][1] if hits else MATERIAL_FAMILY_TEXT_FALLBACK.get(family)
            if value is None: continue
            score, part = part_info(family)
            add(value, score, f"'{n}' = {v}" + (f" ({part})" if part else ""), v + (f" [{part}]" if part else ""))
    for flag, value in MATERIAL_FLAG_ATTRIBUTES.items():            # (b) yes/no flags
        if any(as_bool(v) for v in names.get(norm_label(flag), [])): add(value, 2, f"material flag '{flag}'", f"flag {flag}")
    for field, label in (("shop_material", "Shopify material"), ("material", "product material field")):   # (c) other structured fields
        text = rec.get(field)
        if isinstance(text, str):
            for _, value, kw in find_keywords(text, MATERIAL_KEYWORDS): add(value, 2, f"{label} '{text}'", text)
    return found

def assign_material_akku(rec, stage="evidence"):
    if stage != "evidence": return None
    item, cands = rec["item_id"], []
    found = material_attribute_candidates(item, rec)                                                               # level 1
    if len(found) == 1:
        value, e = next(iter(found.items()))
        cands.append(cand(value, "High", "; ".join(dict.fromkeys(e["notes"]))[:140], "Product attribute"))
    elif len(found) > 1:
        top = max(e["score"] for e in found.values()); tied = [v for v, e in found.items() if e["score"] == top]
        listed = "; ".join(f"{v}: {', '.join(e['shown'][:2])}" for v, e in found.items())
        probs = CLIP_PROBS.get(("akkuleuchten", "material & look"), {}).get(item)
        if len(tied) == 1:
            value = tied[0]; level = "High" if top == 3 else "Medium"
            why = "look-defining part (shade/diffuser)" if top == 3 else ("no part given" if top == 2 else "other materials are less visible parts")
            cands.append(cand(value, level, f"several materials listed: {listed}; {value} chosen: {why}", "Product attribute"))
        elif probs:
            value = max(tied, key=lambda v: probs.get(v, 0))
            cands.append(cand(value, "Medium", f"several materials listed: {listed}; photo model prefers {value} (p={probs.get(value, 0):.2f})", "Product attribute + Image (model)"))
        else:
            value = min(tied, key=lambda v: MATERIAL_PRIORITY.index(v) if v in MATERIAL_PRIORITY else 99)
            cands.append(cand(value, "Medium", f"several materials listed: {listed}; {value} chosen by priority rule (no part given, no photo model)", "Product attribute"))
    text = " ".join(t for t in (rec.get("name_short"), rec.get("variant"), rec.get("colour")) if isinstance(t, str))   # level 2
    hits = find_keywords(text, MATERIAL_KEYWORDS)
    if hits: cands.append(cand(hits[0][1], "Medium", f"material word '{hits[0][2]}' in product text", "Product text"))
    hits = find_keywords(text, MATERIAL_FINISH_KEYWORDS)
    if hits: cands.append(cand(hits[0][1], "Low", f"finish word '{hits[0][2]}' only hints at the material", "Product text"))
    return resolve(cands, "no material in attributes or product text")

# ---------- Light control ----------
def classify_dimming(value):
    f = fold(value).strip()
    if re.search(DIMMING_IGNORE_REGEX, f): return "ignore"
    if re.search(DIMMING_NO_REGEX, f): return "no"
    if re.search(DIMMING_YES_REGEX, f): return "yes"
    return "ignore"

def parse_colour_temperature(values):
    # -> (set of temperatures in K, has non-warm label, has warm label)
    temps, nonwarm, warm = set(), False, False
    for v in values:
        f = fold(v); temps |= {int(t) for t in re.findall(r"(\d{3,4})\s*k", f)}
        nonwarm |= bool(re.search(r"neutral|kalt|tages", f)); warm |= "warm" in f
    return temps, nonwarm, warm

def assign_light_control(rec, stage="evidence"):
    item = rec["item_id"]
    dim_vals = get_attr_all(item, LIGHT_ATTR["dimming"])
    kinds = [classify_dimming(v) for v in dim_vals]
    temps, nonwarm, warm = parse_colour_temperature(get_attr_all(item, LIGHT_ATTR["colour_temperature"]))
    single_warm = len(temps) == 1 and warm and not nonwarm
    if stage == "default":
        if LIGHT_FIXED_DEFAULT and single_warm and not dim_vals:
            return result("Fixed warm white", "Low", f"single warm-white colour temperature ({min(temps)} K) and no dimming information (assumption)", "Inference (default)")
        d = DEFAULT_WHEN_NO_EVIDENCE.get("Light control")
        return result(d, "Low", "no evidence in attributes, text or photo; configured default", "Inference (default)") if d else None
    cands = []
    # ---- level 1: product attributes ----
    if any(as_bool(v) for v in get_attr_all(item, LIGHT_ATTR["remote"])):
        cands.append(cand("Smart (app or remote)", "High", "attribute 'Fernbedienung enthalten' = True", "Product attribute"))
    if len(temps) >= 2:
        span = f"{min(temps)}-{max(temps)} K"
        if nonwarm: cands.append(cand("Tunable white (warm to cool)", "High", f"colour temperature range {span} includes neutral/cool white", "Product attribute"))
        else: cands.append(cand("Tunable white (warm to cool)", "Medium", f"warm-white colour temperature range {span} only", "Product attribute"))
    if "yes" in kinds:
        shown = next(v for v, k in zip(dim_vals, kinds) if k == "yes")
        cands.append(cand("Dimmable", "High", f"dimming attribute: '{shown[:50]}'", "Product attribute"))
    elif "no" in kinds and len(temps) < 2:
        lvl = "High" if single_warm else "Medium"
        cands.append(cand("Fixed warm white", lvl, "dimming attribute says not dimmable" + (f"; single warm-white colour temperature {min(temps)} K" if single_warm else "; colour temperature unknown"), "Product attribute"))
    hits = find_keywords(rec.get("attr_text") or "", LIGHT_KEYWORDS)                         # free text inside other attributes
    seen = set()
    for _, value, kw in hits:
        if value not in seen: seen.add(value); cands.append(cand(value, "Medium", f"keyword '{kw}' in product attributes", "Product attribute"))
    # ---- level 2: product text ----
    text = " ".join(t for t in (rec["name"], rec.get("variant")) if isinstance(t, str))
    seen = set()
    for _, value, kw in find_keywords(text, LIGHT_KEYWORDS):
        if value not in seen: seen.add(value); cands.append(cand(value, "Medium", f"keyword '{kw}' in product text", "Product text"))
    cands.sort(key=lambda c: (tier_of(c["source"]), LIGHT_PRIORITY.index(c["value"]) if c["value"] in LIGHT_PRIORITY else 99))
    return resolve(cands, "no light-control information in attributes or text")

# ---------- Environment ----------
def assign_environment(rec, stage="evidence"):
    item = rec["item_id"]
    if stage == "default":
        d = DEFAULT_WHEN_NO_EVIDENCE.get("Environment")
        return result(d, "Low", "no outdoor hint in attributes, text or photo (inferred from absence)", "Inference (default)") if d else None
    cands = []
    # ---- level 1: product attributes ----
    for v in get_attr_all(item, ENV_ATTR["ip"]):
        m = re.search(r"(?<![a-z0-9])ip\s*-?\s*([0-9x])\s*([0-9x])(?![0-9])", fold(v))
        if m and m.group(2) != "x":
            env = ENV_VALUES[IP_SECOND_DIGIT_TO_ENV.get(int(m.group(2)), "indoor")]
            cands.append(cand(env, "High", f"protection rating '{v}'", "Product attribute"))
    if any(re.search(ENV_SEASON_OUTDOOR_REGEX, fold(v)) for v in get_attr_all(item, ENV_ATTR["season"])):
        cands.append(cand(ENV_VALUES["covered"], "Medium", "assortment flag 'Saisonartikel' = Outdoor", "Product attribute"))
    if rec.get("shop_main_category") in ENV_OUTDOOR_MAIN_CATEGORIES:
        cands.append(cand(ENV_VALUES["covered"], "Medium", "main category is outdoor lighting", "Product attribute"))
    if find_keywords(rec.get("attr_text") or "", {"w": ENV_WEATHER_KEYWORDS}):
        cands.append(cand(ENV_VALUES["full"], "High", "weather-resistance wording in product attributes", "Product attribute"))
    # ---- level 2: product text ----
    text = " ".join(t for t in (rec["name"], rec.get("variant")) if isinstance(t, str))
    if find_keywords(text, {"w": ENV_WEATHER_KEYWORDS}):
        cands.append(cand(ENV_VALUES["full"], "High", "weather-resistance wording in product text", "Product text"))
    hits = find_keywords(text, {"o": ENV_OUTDOOR_KEYWORDS})
    if hits: cands.append(cand(ENV_VALUES["covered"], "Medium", f"outdoor wording '{hits[0][2]}' in product text", "Product text"))
    return resolve(cands, "no environment information in attributes or text")

# registry: (category, attribute) -> function(rec, stage). Add functions here when other categories are switched on.
ASSIGNERS = {
    ("akkuleuchten", "colour group"):    assign_colour_group,
    ("akkuleuchten", "material & look"): assign_material_akku,
    ("akkuleuchten", "light control"):   assign_light_control,
    ("akkuleuchten", "environment"):     assign_environment,
    # (None, "colour group"): assign_colour_group,        # generic fallback for other categories
}
def find_assigner(cat, attr):
    return ASSIGNERS.get((norm_label(cat), norm_label(attr))) or ASSIGNERS.get((None, norm_label(attr)))

In [ ]:
# =====================================================================
# 10. ASSIGN ALL ATTRIBUTES  (label file -> rules: attributes > text -> photo fallback -> default)
# =====================================================================
def transfer_label(cat, attr, item, labelled):
    # value of the most similar labelled product (or weighted vote of the k most similar)
    S_ = SIM[cat]
    if item not in S_.index: return undetermined("product not in similarity matrix"), None
    sims = S_.loc[item, [i for i in labelled.index if i in S_.columns and i != item]].dropna().sort_values(ascending=False)
    if sims.empty: return undetermined("no similarity could be computed to any labelled product"), None
    top = sims.head(SIMILARITY["k"])
    votes = {}
    for i, s_ in top.items(): votes[labelled[i]] = votes.get(labelled[i], 0) + s_ ** 2
    winner = max(votes, key=votes.get)
    best_id = next(i for i in top.index if labelled[i] == winner); best = float(sims[best_id])
    other = sims[[labelled[i] != winner for i in sims.index]]
    margin = best - (float(other.iloc[0]) if len(other) else 0.0)
    if best >= SIMILARITY["high"] and margin >= SIMILARITY["min_margin_for_high"]: level = "High"
    elif best >= SIMILARITY["medium"]: level = "Medium"
    else: level = "Low"
    comps = {k: SIM_PARTS[cat][k].loc[item, best_id] for k in ("text", "structured", "image")}
    used = [k for k, v in comps.items() if not np.isnan(v)]
    comp_txt = ", ".join(f"{k} {v:.2f}" for k, v in comps.items() if not np.isnan(v))
    name_best = name_of.get(best_id) or ""
    evidence = f"most similar labelled product {best_id} '{name_best[:60]}' (similarity {best:.2f}: {comp_txt}; margin {margin:.2f}) has {winner}"
    res = result(winner, level, evidence, "Similarity to labelled products (" + " + ".join(used) + ")")
    neighbours = "; ".join(f"{i}:{labelled[i]}:{s_:.2f}" for i, s_ in sims.head(3).items())
    return res, {"best_id": best_id, "best_similarity": round(best, 3), "margin": round(margin, 3), "top3": neighbours}


def photo_fallback(cat, attr, item, pool, use_model=True):
    # level 3: value from the PHOTO when attributes and text gave nothing. Two photo signals (both optional):
    #   (a) zero-shot image model on the photo          (b) value of the product whose photo looks most similar
    cands, F = [], IMAGE_FALLBACK
    probs = CLIP_PROBS.get((norm_label(cat), norm_label(attr)), {}).get(item) if use_model else None
    if probs:
        v, p = max(probs.items(), key=lambda kv: kv[1])
        if p >= F["clip_min_prob"]:
            cands.append(cand(v, "Medium" if p >= F["clip_medium_prob"] else "Low", f"photo model: looks like {v} (p={p:.2f})", "Image (model)"))
    S_img = SIM_PARTS[cat]["image"]
    if item in S_img.index and pool:
        s_ = S_img.loc[item, [j for j in pool if j in S_img.columns and j != item]].dropna()
        if len(s_):
            best_id = s_.idxmax(); sim = float(s_[best_id])
            if sim >= F["min_image_similarity"]:
                level = "Medium" if sim >= F["medium_image_similarity"] else "Low"
                cands.append(cand(pool[best_id], level, f"photo most similar to product {best_id} '{(name_of.get(best_id) or '')[:45]}' (image similarity {sim:.2f}), which has {pool[best_id]}", "Image (similar product)"))
    cands.sort(key=lambda c: -LEVELS.index(c["level"]))
    return resolve(cands, "") if cands else None

def safe_call(fn, rec, stage, cat, attr):
    try: return fn(rec, stage)
    except Exception as e:
        UNPROCESSED.append((cat, attr, rec["item_id"], f"rule failed: {type(e).__name__}"))
        return undetermined(f"rule failed: {type(e).__name__}") if stage == "evidence" else None

rows, neighbour_rows, UNPROCESSED, loo_jobs, check_rows = [], [], [], [], []
series_of = PRODUCTS.drop_duplicates("item_id").set_index("item_id")["series"].to_dict()
name_of = PRODUCTS.drop_duplicates("item_id").set_index("item_id")["name"].to_dict()
for cat in CATEGORIES:
    targets = PRODUCTS[(PRODUCTS["category"] == cat) & PRODUCTS["is_target"]]
    recs = {}
    for r in targets.itertuples():
        rec = r._asdict(); rec["item_id"] = r.item_id; rec["name"] = r.name; recs[r.item_id] = rec
    labelled_attrs = {a: g.set_index("item_id")["value"] for a, g in LABELS[cat].groupby("attribute")}
    for attr, values in ALLOWED[cat].items():
        fn = find_assigner(cat, attr)
        results = {}
        # ---- pass 1: specialist label / label transfer / rules on attributes and text ----
        for item, rec in recs.items():
            if attr in labelled_attrs and item in labelled_attrs[attr].index:
                res = result(labelled_attrs[attr][item], "High", "assigned by portfolio specialists in label file", "Specialist label file")
            elif attr in labelled_attrs:
                res, nb = transfer_label(cat, attr, item, labelled_attrs[attr])
                if nb: neighbour_rows.append({"item_id": item, "attribute": attr, "assigned": res["value"], **nb})
            elif fn is not None:
                res = safe_call(fn, rec, "evidence", cat, attr)
            else:
                res = undetermined("no label file and no rule implemented for this attribute"); UNPROCESSED.append((cat, attr, item, "no assigner"))
            results[item] = res
        if attr in labelled_attrs: loo_jobs.append((cat, attr, labelled_attrs[attr]))
        if fn is not None and attr not in labelled_attrs:
            pool = {i: r_["value"] for i, r_ in results.items() if r_["confidence"] in ("High", "Medium") and r_["value"] != NOT_DETERMINABLE}
            # ---- pass 2: photo fallback for products still without a value ----
            if IMAGE_FALLBACK["enabled"]:
                F = IMAGE_FALLBACK; S_img = SIM_PARTS[cat]["image"]
                # first test how reliable the photo signal is on products that HAVE an attribute/text value
                sims_ok = []
                for i, v in pool.items():
                    if i not in S_img.index: continue
                    s_ = S_img.loc[i, [j for j in pool if j in S_img.columns and j != i]].dropna()
                    if len(s_) and float(s_.max()) >= F["min_image_similarity"]: sims_ok.append((float(s_.max()), pool[s_.idxmax()] == v))
                base = float(pd.Series(list(pool.values())).value_counts(normalize=True).iloc[0]) if pool else np.nan
                if pool: check_rows.append({"category": cat, "attribute": attr, "photo signal": "baseline: always the most common value", "products tested": len(pool), "value reproduced": round(base, 2), "used": ""})
                use_similar = bool(sims_ok) and (not F["require_beat_baseline"] or float(np.mean([x[1] for x in sims_ok])) > base)
                if sims_ok:
                    arr = np.array(sims_ok, dtype=float)
                    for lab_, mask in ((f"similar photo, {F['min_image_similarity']:.2f}-{F['medium_image_similarity']:.2f}", arr[:, 0] < F["medium_image_similarity"]), (f"similar photo, >= {F['medium_image_similarity']:.2f}", arr[:, 0] >= F["medium_image_similarity"])):
                        if mask.any(): check_rows.append({"category": cat, "attribute": attr, "photo signal": lab_, "products tested": int(mask.sum()), "value reproduced": round(float(arr[mask, 1].mean()), 2), "used": "yes" if use_similar else "no (not better than baseline)"})
                probs_all = CLIP_PROBS.get((norm_label(cat), norm_label(attr)))
                use_model = False
                if probs_all:
                    hits_ = [max(probs_all[i], key=probs_all[i].get) == v for i, v in pool.items() if i in probs_all]
                    if hits_:
                        use_model = (not F["require_beat_baseline"]) or float(np.mean(hits_)) > base
                        check_rows.append({"category": cat, "attribute": attr, "photo signal": "image model (zero-shot)", "products tested": len(hits_), "value reproduced": round(float(np.mean(hits_)), 2), "used": "yes" if use_model else "no (not better than baseline)"})
                for item in [i for i, r_ in results.items() if r_["value"] == NOT_DETERMINABLE]:
                    res = photo_fallback(cat, attr, item, pool if use_similar else {}, use_model)
                    if res is not None and res["value"] != NOT_DETERMINABLE: results[item] = res
            # ---- pass 3: optional default (level 4) ----
            for item, rec in recs.items():
                if results[item]["value"] == NOT_DETERMINABLE:
                    d = safe_call(fn, rec, "default", cat, attr)
                    if d is not None and d["value"] != NOT_DETERMINABLE: results[item] = d
        # ---- every value must be one of the allowed values (or the placeholder) ----
        for item, res in results.items():
            if res["value"] != NOT_DETERMINABLE:
                canon = canonical_value(res["value"], values)
                if canon is None:
                    UNPROCESSED.append((cat, attr, item, f"value '{res['value']}' not allowed"))
                    res = undetermined(f"rule produced '{res['value']}' which is not in the allowed list")
                else: res["value"] = canon
            rows.append({"item_id": item, "product_name": recs[item]["name"], "subcategory": cat, "attribute": attr,
                         "assigned_value": res["value"], "confidence": res["confidence"],
                         "evidence": res["evidence"], "evidence_source": res["source"]})
assignments = pd.DataFrame(rows)
neighbours = pd.DataFrame(neighbour_rows)
IMAGE_CHECK = pd.DataFrame(check_rows)
if len(neighbours): neighbours.to_csv(INTERIM_DIR / "label_transfer_neighbours.csv", index=False, sep=CSV_SEP, encoding=CSV_ENCODING)
issue("attribute assignments that could not be processed", len(UNPROCESSED),
      "; ".join(sorted({f"{a}/{b}: {d}" for a, b, c, d in UNPROCESSED}))[:300])
COUNTS["assignment_rows"] = len(assignments)

# ---- leave-one-out check: how well does "label of the most similar OTHER labelled product" reproduce the specialists' labels? ----
def loo_predictions(S_, lab_s, exclude_same_series=False):
    out = []
    for i in lab_s.index:
        if i not in S_.index: continue
        s_ = S_.loc[i, [j for j in lab_s.index if j != i and j in S_.columns]].dropna()
        if exclude_same_series and isinstance(series_of.get(i), str):
            s_ = s_.drop([j for j in s_.index if series_of.get(j) == series_of.get(i)])
        if len(s_): out.append((i, float(s_.max()), lab_s[s_.idxmax()] == lab_s[i]))
    return pd.DataFrame(out, columns=["item_id", "similarity", "correct"])

loo_rows = []
for cat, attr, lab_s in loo_jobs:
    for name, S_ in [("combined", SIM[cat])] + [(f"{k} only", v) for k, v in SIM_PARTS[cat].items()]:
        p_all, p_ex = loo_predictions(S_, lab_s), loo_predictions(S_, lab_s, True)
        loo_rows.append({"category": cat, "attribute": attr, "similarity used": name, "labelled products": len(p_all),
                         "label reproduced": round(p_all["correct"].mean(), 2) if len(p_all) else np.nan,
                         "excl. same series": round(p_ex["correct"].mean(), 2) if len(p_ex) else np.nan})
    p = loo_predictions(SIM[cat], lab_s)
    for lo, hi, nm in ((0, SIMILARITY["medium"], "Low band"), (SIMILARITY["medium"], SIMILARITY["high"], "Medium band"), (SIMILARITY["high"], 1.01, "High band")):
        q = p[(p["similarity"] >= lo) & (p["similarity"] < hi)]
        loo_rows.append({"category": cat, "attribute": attr, "similarity used": f"combined, {nm} ({lo:.2f}-{min(hi, 1):.2f})",
                         "labelled products": len(q), "label reproduced": round(q["correct"].mean(), 2) if len(q) else np.nan, "excl. same series": np.nan})
LOO_TABLE = pd.DataFrame(loo_rows)
log(f"{len(assignments)} assignment rows created", show=True)

In [ ]:
# =====================================================================
# 11. LOAD AND AGGREGATE SALES  (one row per product; missing sales are NOT zero sales)
# =====================================================================
targets_all = PRODUCTS[PRODUCTS["is_target"]].drop_duplicates("item_id")
SALES_INFO = {"status": "no sales file", "date_column": None, "available_range": None, "rows_used": 0}
sales_summary = targets_all[["item_id", "name", "category"]].rename(columns={"name": "product_name", "category": "subcategory"}).reset_index(drop=True)
sales_summary["in_sales_file"] = False
measure_cols, source_totals = [], {}

if "sales" in TABLES:
    T = TABLES["sales"]; sdf_raw = T["df"]; scol = T["cols"]
    key = sdf_raw[scol["item_id"]].map(norm_key)
    valid = key.fillna("").str.fullmatch(r"-?\d+")
    issue("sales rows without numeric item key (e.g. 'Total' row, ignored)", (~valid).sum())
    sdf = pd.DataFrame({"item_id": key[valid]})
    lookup = {norm_name(c): c for c in sdf_raw.columns}
    aggs = {}
    for out, (cands_, how) in SALES_MEASURES.items():
        col = next((lookup[norm_name(c)] for c in cands_ if norm_name(c) in lookup), None)
        if col is None: issue("sales measure column not found", 1, out); continue
        sdf[out] = sdf_raw.loc[valid, col].map(parse_number).values; aggs[out] = how
    # optional date / order columns
    date_col, order_col = scol.get("date"), scol.get("order_id")
    if order_col: sdf["_order_id"] = sdf_raw.loc[valid, order_col].values
    if date_col:
        sdf["_date"] = pd.to_datetime(sdf_raw.loc[valid, date_col], dayfirst=True, errors="coerce").values
        issue("sales rows with unreadable date", sdf["_date"].isna().sum())
        lo, hi = sdf["_date"].min(), sdf["_date"].max()
        start, end = pd.Timestamp(SALES_PERIOD[0]), pd.Timestamp(SALES_PERIOD[1])
        SALES_INFO.update(date_column=date_col, available_range=f"{lo:%Y-%m-%d} to {hi:%Y-%m-%d}")
        if hi >= start and lo <= end:
            sdf = sdf[(sdf["_date"] >= start) & (sdf["_date"] <= end)]; SALES_INFO["status"] = "period filter applied"
        else:
            SALES_INFO["status"] = "requested period NOT in data - all dates used"
            issue("requested sales period not present in source", 1, f"available {SALES_INFO['available_range']}")
    else:
        SALES_INFO["status"] = "no date column - period filter NOT applicable (all rows used)"
        issue("sales file has no date column", 1, f"cannot restrict to {SALES_PERIOD[0]} .. {SALES_PERIOD[1]}; totals are undated")
    sdf = sdf[sdf["item_id"].isin(set(targets_all["item_id"]))]
    SALES_INFO["rows_used"] = len(sdf)
    # aggregate FIRST (one row per product), then join 1:1
    agg_spec = {m: (m, how) for m, how in aggs.items()}
    agg_spec["n_sales_rows"] = ("item_id", "size")
    if order_col: agg_spec["n_orders"] = ("_order_id", "nunique")
    if date_col: agg_spec.update(first_sale_date=("_date", "min"), last_sale_date=("_date", "max"))
    sales_agg = sdf.groupby("item_id").agg(**agg_spec).reset_index()
    issue("products with several sales rows (summed)", (sales_agg["n_sales_rows"] > 1).sum())
    measure_cols = list(aggs)
    source_totals = {m: float(sdf[m].sum()) for m in measure_cols if aggs[m] == "sum"}
    sales_summary = sales_summary.merge(sales_agg, on="item_id", how="left", validate="one_to_one")
    sales_summary["in_sales_file"] = sales_summary["item_id"].isin(sales_agg["item_id"])
    # ranks and share inside the sub-category (only among products that appear in the sales file)
    for m, name in ((SALES_PRIMARY_REVENUE, "rank_revenue_in_subcategory"), (SALES_PRIMARY_UNITS, "rank_units_in_subcategory")):
        if m in sales_summary: sales_summary[name] = sales_summary.groupby("subcategory")[m].rank(method="min", ascending=False)
    if SALES_PRIMARY_REVENUE in sales_summary:
        tot = sales_summary.groupby("subcategory")[SALES_PRIMARY_REVENUE].transform("sum")
        sales_summary["revenue_share_pct_in_subcategory"] = (100 * sales_summary[SALES_PRIMARY_REVENUE] / tot).round(2)
    n_in = int(sales_summary["in_sales_file"].sum())
    issue("target products not found in sales file (sales left empty, not zero)", len(sales_summary) - n_in)
    log(f"Sales: {n_in}/{len(sales_summary)} target products in sales file; {SALES_INFO['status']}", show=True)
    sales_doc = {"product key": f"{T['cols']['item_id']} (numeric ids; non-numeric rows such as 'Total' dropped)",
                 "date column": date_col or "none in source", "period requested": f"{SALES_PERIOD[0]} to {SALES_PERIOD[1]}",
                 "period status": SALES_INFO["status"], "available date range": SALES_INFO["available_range"] or "n/a (no date column)",
                 "measures (column -> output)": "; ".join(f"{(next((c for c in SALES_MEASURES[m][0] if norm_name(c) in lookup), '?'))} -> {m} [{aggs[m]}]" for m in measure_cols),
                 "ranking measure": f"{SALES_PRIMARY_REVENUE} (units: {SALES_PRIMARY_UNITS})", "currency": SALES_CURRENCY,
                 "orders / first+last sale date": "not available in source" if not date_col else "from date column",
                 "filters": "only selected products; no other filters"}
    for k, v in sales_doc.items(): log(f"sales rule - {k}: {v}")
else:
    log("No sales file - sales columns will be empty", "WARN", show=True)

In [ ]:
# =====================================================================
# 12. WIDE PRODUCT TABLE (+ sales)  and  13. VALIDATION
# =====================================================================
# ---- wide table: one row per product, one column per attribute (+ confidence/source) ----
base_cols = ["item_id", "name", "category", "manufacturer_name", "series", "product_type", "variant", "colour",
             "length", "depth", "height", "weight", "image_status"] + list(EXTRA_ROLES)
wide = targets_all[[c for c in base_cols if c in targets_all]].rename(columns={"name": "product_name", "category": "subcategory"}).reset_index(drop=True)
for attr in dict.fromkeys(assignments["attribute"]):
    a = assignments[assignments["attribute"] == attr].set_index(["item_id", "subcategory"])
    idx = pd.MultiIndex.from_frame(wide[["item_id", "subcategory"]])
    wide[attr] = a["assigned_value"].reindex(idx).values
    wide[f"{attr} | confidence"] = a["confidence"].reindex(idx).values
    wide[f"{attr} | source"] = a["evidence_source"].reindex(idx).values
    if len(neighbours) and attr in set(neighbours["attribute"]):
        nb = neighbours[neighbours["attribute"] == attr].set_index("item_id")
        wide[f"{attr} | most similar labelled item"] = wide["item_id"].map(nb["best_id"])
        wide[f"{attr} | similarity"] = wide["item_id"].map(nb["best_similarity"])

wide_sales = wide.merge(sales_summary.drop(columns=["product_name", "subcategory"]), on="item_id", how="left", validate="one_to_one")

# ---- validation checks ----
CHECKS = []
def check(name, ok, detail="", level="FAIL"):
    CHECKS.append({"type": "validation check", "topic": name, "status": "PASS" if ok else level, "count": "", "detail": "" if ok else detail})

check("target product keys are unique", not selected["item_id"].duplicated().any(), f"{int(selected['item_id'].duplicated().sum())} duplicated ids")
expected = {(i, c, a) for c, g in selected.groupby("category") for i in g["item_id"] for a in ALLOWED[c]}
actual = set(zip(assignments["item_id"], assignments["subcategory"], assignments["attribute"]))
check("every product has a row for every required attribute", expected <= actual, f"{len(expected - actual)} missing rows")
check("no duplicated product/attribute rows", not assignments.duplicated(["item_id", "subcategory", "attribute"]).any(),
      f"{int(assignments.duplicated(['item_id', 'subcategory', 'attribute']).sum())} duplicates")
invalid = [(r.item_id, r.attribute, r.assigned_value) for r in assignments.itertuples()
           if r.assigned_value != NOT_DETERMINABLE and r.assigned_value not in ALLOWED[r.subcategory][r.attribute]]
check("every assigned value is allowed for its sub-category", not invalid, f"{len(invalid)} invalid, e.g. {invalid[:3]}")
check("confidence levels are valid", assignments["confidence"].isin(LEVELS).all(), "unknown confidence label")
check("every row has evidence and source", assignments["evidence"].astype(bool).all() and assignments["evidence_source"].astype(bool).all(), "empty evidence")
for cat in CATEGORIES:
    for attr, g in LABELS[cat].groupby("attribute"):
        got = assignments[(assignments["subcategory"] == cat) & (assignments["attribute"] == attr)].set_index("item_id")["assigned_value"]
        lab_t = g.set_index("item_id")["value"]; common = lab_t.index.intersection(got.index)
        check(f"{cat}/{attr}: labelled products keep their specialist value", (got[common] == lab_t[common]).all(), "label overwritten")
check("wide table has one row per selected product", len(wide) == len(targets_all) and not wide["item_id"].duplicated().any(), f"{len(wide)} rows vs {len(targets_all)} products")
check("sales join did not change row count", len(wide_sales) == len(wide), f"{len(wide_sales)} vs {len(wide)}")
if source_totals:
    joined = {m: float(wide_sales[m].sum(skipna=True)) for m in source_totals}
    diff = {m: (joined[m], source_totals[m]) for m in source_totals if not np.isclose(joined[m], source_totals[m])}
    check("sales totals are not duplicated by the join", not diff, f"differences (joined, source): {diff}")
check("sales period could be applied", SALES_INFO["status"].startswith("period filter"), SALES_INFO["status"], level="WARN")
check("all target products have a usable image", (wide["image_status"] == "ok").all(), f"{int((wide['image_status'] != 'ok').sum())} without usable image", level="WARN")
for attr, share in assignments.assign(nd=assignments["assigned_value"] == NOT_DETERMINABLE).groupby(["subcategory", "attribute"])["nd"].mean().items():
    check(f"{attr[0]}/{attr[1]}: share of '{NOT_DETERMINABLE}' <= {MAX_NOT_DETERMINABLE_SHARE:.0%}", share <= MAX_NOT_DETERMINABLE_SHARE, f"{share:.0%} not determinable", level="WARN")
validation = pd.concat([pd.DataFrame(CHECKS), pd.DataFrame(ISSUES).assign(type="data quality issue", status="INFO").rename(columns={"topic": "topic"})[["type", "topic", "status", "count", "detail"]]], ignore_index=True)
validation.to_csv(RES_DIR / OUTPUT_FILES["validation"], index=False, sep=CSV_SEP, encoding=CSV_ENCODING)
n_fail, n_warn = sum(c["status"] == "FAIL" for c in CHECKS), sum(c["status"] == "WARN" for c in CHECKS)
log(f"Validation: {len(CHECKS) - n_fail - n_warn} passed, {n_warn} warnings, {n_fail} failed", show=True)
if n_fail or n_warn: display(pd.DataFrame(CHECKS).query("status != 'PASS'")[["status", "topic", "detail"]].reset_index(drop=True))
if n_fail and STRICT_VALIDATION: raise AssertionError(f"{n_fail} validation check(s) failed - see {OUTPUT_FILES['validation']}")

In [ ]:
# =====================================================================
# 14. EXPORT RESULT FILES + PROCESSING LOG
# =====================================================================
def save(df, key):
    df.to_csv(RES_DIR / OUTPUT_FILES[key], index=False, sep=CSV_SEP, encoding=CSV_ENCODING)
save(assignments, "detail"); save(wide, "wide"); save(sales_summary, "sales"); save(wide_sales, "wide_sales")
review = assignments[assignments["confidence"].isin(["Low", "Not determinable"])]
review.to_csv(INTERIM_DIR / "assignments_to_review.csv", index=False, sep=CSV_SEP, encoding=CSV_ENCODING)

with open(RES_DIR / OUTPUT_FILES["log"], "w", encoding="utf-8") as f:
    f.write("CATEGORY MATRIX - PROCESSING LOG\n" + "=" * 60 + "\n")
    f.write(f"run: {time.strftime('%Y-%m-%d %H:%M:%S')}  | categories: {', '.join(CATEGORIES)}\n\nROW COUNTS\n")
    for k, v in COUNTS.items(): f.write(f"  {k}: {v}\n")
    f.write(f"  products_in_wide_table: {len(wide)}\n  review_rows (Low / Not determinable): {len(review)}\n")
    f.write("\nLEAVE-ONE-OUT CHECK OF LABEL TRANSFER\n" + (LOO_TABLE.to_string(index=False) if len(LOO_TABLE) else "  no labels") + "\n")
    if len(IMAGE_CHECK): f.write("\nPHOTO FALLBACK CHECK\n" + IMAGE_CHECK.to_string(index=False) + "\n")
    f.write("\nSKIPPED RECORDS, UNRESOLVED JOINS, FAILED IMAGES, LIMITATIONS\n")
    for i in ISSUES: f.write(f"  - {i['topic']}: {i['count']} {i['detail']}\n")
    f.write("\nDETAILED LOG\n"); f.write("\n".join(LOG) + "\n")
log(f"Files written to {RES_DIR}: " + ", ".join(OUTPUT_FILES.values()), show=True)

In [ ]:
# =====================================================================
# 15. SUMMARY TABLES
# =====================================================================
print("Products per sub-category"); display(selected.groupby("category").size().rename("products").to_frame().T)

valid_mask = assignments["assigned_value"] != NOT_DETERMINABLE
comp = assignments.assign(completed=valid_mask).groupby(["subcategory", "attribute"], sort=False)["completed"].agg(["sum", "size"])
comp["completed %"] = (100 * comp["sum"] / comp["size"]).round(1)
print("\nCompleted assignments by attribute"); display(comp.rename(columns={"sum": "completed", "size": "products"}))

print("Confidence distribution (rows)")
display(pd.crosstab([assignments["subcategory"], assignments["attribute"]], assignments["confidence"]).reindex(columns=[l for l in reversed(LEVELS)], fill_value=0))

def evidence_group(src):
    if src.startswith("Specialist"): return "Specialist label"
    if src.startswith("Similarity"): return "Label transfer"
    if src.startswith("Inference"): return "4 Default"
    if src.startswith("No reliable"): return "none"
    if src.startswith("Product attribute"): return "1 Attribute" + (" (+ confirmed)" if "+" in src else "")
    if src.startswith("Product text"): return "2 Text" + (" (+ confirmed)" if "+" in src else "")
    return "3 Photo"
print("\nWhich evidence was used (rows) - attributes have priority over text, text over photo")
display(pd.crosstab([assignments["subcategory"], assignments["attribute"]], assignments["evidence_source"].map(evidence_group)))

if len(IMAGE_CHECK):
    print("How reliable is the photo signal? (tested on products that have an attribute/text based value)")
    display(IMAGE_CHECK.drop(columns=["category"]))

print("Missing or uncertain assignments (Low / Not determinable): products per attribute")
unc = review.groupby(["attribute", "confidence"])["item_id"].nunique().unstack(fill_value=0)
display(unc)

if sales_summary["in_sales_file"].any():
    print(f"Top sellers per sub-category - {SALES_PRIMARY_REVENUE} ({SALES_INFO['status']})")
    top = (wide_sales[wide_sales["in_sales_file"]].sort_values(SALES_PRIMARY_REVENUE, ascending=False).groupby("subcategory").head(5))
    show_cols = ["subcategory", "item_id", "product_name", SALES_PRIMARY_UNITS, SALES_PRIMARY_REVENUE, "rank_revenue_in_subcategory"] + [c for c in wide_sales if c in ("Design style",)]
    display(top[show_cols].reset_index(drop=True))

if len(LOO_TABLE):
    print("Check of the label transfer (leave-one-out on the labelled products)")
    display(LOO_TABLE.drop(columns=["category", "attribute"]))

if UNPROCESSED:
    print("Attributes / values that could not be processed")
    display(pd.DataFrame(UNPROCESSED, columns=["category", "attribute", "item_id", "reason"]).groupby(["category", "attribute", "reason"]).size().rename("products").reset_index())

# README

## How to run
1. Folder layout: `Category Matrix/` (this notebook) · `SRC/` (source files, incl. `DesignStyle_Akkuleuchte.csv`) · `RES/` (created automatically).
2. Python packages: `pandas`, `numpy`, `requests`, `Pillow`, `scikit-learn` (required). `torch` + `transformers` are **optional** (only for `USE_CLIP = True`).
3. Internet access is needed once to download product images from `ItemMedia.csv` (cached in `RES/cache_images`, re-runs reuse the cache) and, if `USE_CLIP = True`, once to download the image model. Without internet / without `torch` + `transformers` the notebook still runs: it then uses simple shape/colour features and skips the zero-shot model.
4. Run all cells (**Kernel → Restart & Run All**). Settings are only in Section 0.
5. To process more sub-categories remove the `#` in `CATEGORIES`. Attributes without label file or rule are filled with the placeholder and listed under "could not be processed" until a rule is added to `ASSIGNERS`.
6. To add new information (tier, price, ...): put the file in `SRC` and fill `EXTRA_PRODUCT_FILES` (or `EXTRA_ATTRIBUTE_FEATURES`). Role `numeric` / `categorical` / `text` makes it part of the similarity, `info` only adds it to the outputs.

## Files in `RES`
| File | Content |
|---|---|
| `assignments_detailed.csv` | one row per product × attribute: value, confidence, evidence, evidence source |
| `products_wide.csv` | one row per product, one column per attribute (+ confidence, source, most similar labelled item) |
| `sales_summary_by_product.csv` | sales measures per product, rank and revenue share inside the sub-category |
| `products_wide_with_sales.csv` | wide table joined 1:1 with the sales summary |
| `validation_report.csv` | validation checks (PASS/WARN/FAIL) and data-quality issues |
| `processing_log.txt` | row counts, skipped records, failed images, unresolved joins, limitations, sales rules |
| `cache_images/`, `intermediate/` | cached images, image download log, nearest neighbours of the label transfer, rows to review |

## Evidence priority (applies to Colour Group, Material & look, Light control, Environment)
| Level | Evidence | Rule |
|---|---|---|
| 1 | **Product attributes** (`ItemAttribute.csv`, structured product fields) | always win – text or photo can never overrule them |
| 2 | Product text (name, variant) | only if level 1 gives nothing for this attribute |
| 3 | **Product photo** (zero-shot image model, or the product with the most similar photo) | only if levels 1 and 2 give nothing |
| 4 | Optional default (`DEFAULT_WHEN_NO_EVIDENCE`) | only if levels 1-3 give nothing; always *Low* |

If a lower level independently names the **same** value as the winning one, the confidence goes up by one level (shown as `Product attribute + Product text` in the source column). Values from lower levels that disagree are listed in the evidence text as "Not used".

## Method per attribute (Akkuleuchten)
* **Design style** – products in `DesignStyle_Akkuleuchte.csv` keep their specialist value (*High*). All others receive the value of the **most similar labelled product** (`k = 1`). Similarity = weighted mean of *text* (TF-IDF on name without colour words, series, manufacturer), *structured* (dimensions, weight, manufacturer, product type, series, extra columns) and *image* (silhouette/edge shape, or CLIP embeddings if enabled). A leave-one-out check on the labelled products is printed and logged.
* **Material & look** – attributes `Material (Metall / Kunststoff / Mineralien / Holz / Textil ...)`, the flags `Metall`, `Kunststoff`, `Glas` ..., Shopify material and the product material field. One material → *High*. **Several materials** (e.g. steel + polycarbonate) → the part decides (`Materialzusatz`: shade/diffuser beats no part beats base/frame); if still tied, the photo model decides (if enabled), otherwise `MATERIAL_PRIORITY`. Result *Medium*; the evidence text lists all materials found.
* **Light control** – attributes `Dimmbar (Leuchten / Leuchtmittel)`, `Lichtfarbe, Farbtemperatur` (a range such as 2700-4000 K = tunable white; a warm-only range is only a *Medium* signal), `Fernbedienung enthalten` = True (smart). Several functions → the more capable one wins (`LIGHT_PRIORITY`) within the same confidence. 'Nein' / False + single warm-white colour temperature → *Fixed warm white*. Text keywords only count if there is no attribute evidence.
* **Environment** – `Schutzart` (IP code, second digit: 0 → indoor, 1-3 → covered outdoor, 4+ → full outdoor; table in the config), `Saisonartikel` = Outdoor and main category *Außenleuchten* (both *Medium*), weather wording. Then product text, photo, and finally the default *Indoor only* (*Low*).
* **Colour Group** – Shopify colour / colour name / variant (attributes), then colour words in the product name, then the dominant colour of the photo. Several groups named together → `Mix`.
* **Photo fallback** – (a) zero-shot image model (`IMAGE_PROMPTS`, currently for Material & look) and (b) the value of the product whose photo is most similar (and that has an attribute/text value). Similarity ≥ `medium_image_similarity` → *Medium*, ≥ `min_image_similarity` → *Low*. The summary table "How reliable is the photo signal?" tests both on products that already have a value and shows the baseline "always the most common value" – with `require_beat_baseline = True` a photo signal is **only used for an attribute if it beats that baseline** (column "used"). A photo cannot show dimming or IP rating; for Light control and Environment the photo signal is only an indirect hint (similar model/variant).

## Confidence
| Level | Meaning |
|---|---|
| High | specialist label; an attribute that states the value directly; explicit IP code; or two evidence levels agree; label transfer with similarity ≥ `high` and a clear margin |
| Medium | partial attribute evidence (e.g. one of several materials chosen by rule), value from product text, strong photo similarity / model result, label transfer with similarity ≥ `medium` |
| Low | weak or indirect evidence (finish word, image colour, weak photo similarity, default by absence of evidence) |
| Not determinable | no valid evidence – value is the placeholder `Not determinable` |

The placeholder is **not** part of the allowed value list; validation accepts it only as an explicit "no value" marker.

## Sales rules
Product key `Item (group)` (numeric ids; the `Total` row is dropped). Measures: Net order intake I/II (value), order item qty, NOI II qty, stock (snapshot, not summed). Ranking by `Net order intake II` within the sub-category. **The current sales file has no date column**, so the period Nov 2025 – Oct 2026 cannot be applied: totals are undated, first/last sale date and number of orders are not available. If a date column (and optionally an order id) is added later, the period filter, dates and order counts are applied automatically. Products missing in the sales file stay empty (not 0). Currency is not stated in the file.

## Assumptions and limitations
* Label transfer reproduces the specialists' logic only as far as text, dimensions and image shape reflect design style; check the logged leave-one-out accuracy and the Low/Medium rows in `intermediate/assignments_to_review.csv`.
* Variants of one model (colours) share name/series, so they usually inherit the label of a labelled variant – this is intended.
* Image features without CLIP are simple (silhouette, edges, colour) and work best on packshots with plain backgrounds.
* Keyword dictionaries are starting points; extend them in Section 0.11.
* `Designstyles.pdf` is no longer used.
* `ItemAttribute.csv`: a product can list the same attribute several times (e.g. two sets with `Material (Kunststoff)`); all distinct values are kept. `Null`, `0`, `-` count as missing.
* IP44 is treated as *Full outdoor* (usual minimum for garden luminaires). Change `IP_SECOND_DIGIT_TO_ENV` if the portfolio team sees it differently.